
## Real-Time Intelligent Customer Support Using Streaming, Delta Lakehouse, Data Quality, Lineage, and Advanced RAG

### End-to-end architecture

```text
Bitext Customer-Support Dataset
            ↓
Kafka-style Producer / Consumer
            ↓
Pydantic Data Contract + Quarantine
            ↓
Delta Lakehouse: Bronze → Silver → Gold
            ↓
Great Expectations + DAMA Quality Report
            ↓
NovaStore Knowledge Base
            ↓
Chunking → Embeddings → ChromaDB + BM25
            ↓
RRF Hybrid Search → Cross-Encoder Reranking
            ↓
Llama-Generated Grounded Answer with Sources
            ↓
OpenLineage Events + Airflow DAG Artifact
```

>

## Step 1 — Install Dependencies

Run the installation cell **once only** in a fresh Google Colab runtime.

This version deliberately does **not** reinstall Colab's preinstalled scientific stack
(`numpy`, `pandas`, `pyarrow`, `scipy`, and `scikit-learn`). Reinstalling those packages
inside a live runtime can create binary incompatibility errors.

After the installation finishes:

1. Choose **Runtime → Restart session** once.
2. Do not run the installation cell again.
3. Run the notebook from the first import/configuration cell downward.

The project packages include:

- `datasets` and `pydantic` for ingestion and validation;
- `pyspark` and `delta-spark` for the Lakehouse;
- `great_expectations` for quality validation;
- `chromadb`, `sentence-transformers`, and `rank-bm25` for Advanced RAG;
- `groq` for optional Llama generation.


In [3]:


%pip install -q --no-cache-dir \
    datasets \
    loguru \
    "pydantic>=2.0,<3.0" \
    "pyspark==4.0.1" \
    "delta-spark==4.0.0" \
    "great_expectations==0.18.21" \
    chromadb \
    sentence-transformers \
    rank-bm25 \
    confluent-kafka \
    openlineage-python \
    groq

print("Project dependencies installed.")
print("Now restart the session once, then run the notebook from the top.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 123.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 331.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 294.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 241.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 281.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 206.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 315.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 337.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 349.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 436.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 354.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 311.6 MB/s eta 0:00:0

### Important Runtime Order

After restarting the session, start from the next cell and continue in order.
Do not jump directly to a RAG, Spark, or Llama cell because configuration objects
such as `PATHS`, `EMBEDDING_MODEL`, and `KB_CHUNKS` are created earlier.


## Step 2 — Imports and Global Configuration

This cell imports all libraries, fixes random seeds for reproducibility, and creates the project directory structure.

The default sample is deliberately small because the final project has a two-day implementation window. Increase `SAMPLE_SIZE` after the complete pipeline works successfully.

In [1]:
from loguru import logger
from datasets import load_dataset
from pydantic import BaseModel

print("Required packages imported successfully.")

Required packages imported successfully.


In [2]:
import sys
import numpy as np
import pandas as pd
import pyarrow as pa
import sklearn
import chromadb
import sentence_transformers

print("Python              :", sys.version.split()[0])
print("NumPy              :", np.__version__)
print("Pandas             :", pd.__version__)
print("PyArrow            :", pa.__version__)
print("Scikit-learn       :", sklearn.__version__)
print("ChromaDB           :", chromadb.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("Environment verification passed.")


Python              : 3.12.13
NumPy              : 1.26.4
Pandas             : 2.2.2
PyArrow            : 18.1.0
Scikit-learn       : 1.6.1
ChromaDB           : 1.5.9
SentenceTransformers: 5.6.0
Environment verification passed.


In [3]:
import asyncio
import json
import os
import random
import re
import shutil
import uuid
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
from datasets import load_dataset
from loguru import logger
from pydantic import BaseModel, ConfigDict, Field, field_validator

# ---------------------------------------------------------------------
# REPRODUCIBILITY
# ---------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------
# PROJECT CONFIGURATION
# ---------------------------------------------------------------------
DATASET_ID = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
SAMPLE_SIZE = 200
STREAM_DELAY_SECONDS = 0.01
NUM_KAFKA_PARTITIONS = 3

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GENERATION_PROVIDER = "groq"
GENERATION_MODEL = "llama-3.1-8b-instant"
ENABLE_LLM_GENERATION = True
GENERATION_TEMPERATURE = 0
GENERATION_MAX_TOKENS = 180

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/supportai_final_project")
else:
    PROJECT_ROOT = Path("./supportai_final_project")

PATHS = {
    "root": PROJECT_ROOT,
    "raw": PROJECT_ROOT / "data" / "raw",
    "bronze": PROJECT_ROOT / "data" / "bronze",
    "silver": PROJECT_ROOT / "data" / "silver",
    "gold": PROJECT_ROOT / "data" / "gold",
    "quarantine": PROJECT_ROOT / "data" / "quarantine",
    "knowledge_base": PROJECT_ROOT / "knowledge_base",
    "vector_db": PROJECT_ROOT / "vector_db",
    "quality": PROJECT_ROOT / "quality_reports",
    "lineage": PROJECT_ROOT / "lineage_events",
    "dags": PROJECT_ROOT / "dags",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

logger.remove()
logger.add(lambda message: print(message, end=""), format="{time:HH:mm:ss} | {level:<8} | {message}")
logger.add(PATHS["root"] / "pipeline_execution.log",
           format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")

print("=" * 70)
print("  SUPPORTAI PROJECT CONFIGURATION")
print("=" * 70)
print(f"Dataset             : {DATASET_ID}")
print(f"Sample size         : {SAMPLE_SIZE}")
print(f"Kafka partitions    : {NUM_KAFKA_PARTITIONS}")
print(f"Embedding model     : {EMBEDDING_MODEL}")
print(f"Cross-encoder       : {RERANKER_MODEL}")
print(f"Generation provider : {GENERATION_PROVIDER}")
print(f"Generation model    : {GENERATION_MODEL}")
print(f"LLM generation      : {ENABLE_LLM_GENERATION}")
print(f"Project directory   : {PROJECT_ROOT.resolve()}")
print("=" * 70)

  SUPPORTAI PROJECT CONFIGURATION
Dataset             : bitext/Bitext-customer-support-llm-chatbot-training-dataset
Sample size         : 200
Kafka partitions    : 3
Embedding model     : sentence-transformers/all-MiniLM-L6-v2
Cross-encoder       : cross-encoder/ms-marco-MiniLM-L-6-v2
Generation provider : groq
Generation model    : llama-3.1-8b-instant
LLM generation      : True
Project directory   : /content/supportai_final_project


### Step 2.1 — Configure Groq for Llama Generation

Before running the RAG demonstration, add `GROQ_API_KEY` to **Google Colab Secrets**.

The notebook does not display or save the key. If no key is configured, the project still completes using the extractive grounded fallback.


In [4]:
# =====================================================================
# OPTIONAL CHECK — GROQ KEY AVAILABILITY
# =====================================================================

groq_key_available = bool(os.getenv("GROQ_API_KEY"))

if IN_COLAB and not groq_key_available:
    try:
        from google.colab import userdata
        groq_key_available = bool(userdata.get("GROQ_API_KEY"))
    except Exception:
        groq_key_available = False

print("Groq API key available:", groq_key_available)
print(
    "Generation mode:",
    f"Groq Llama ({GENERATION_MODEL})"
    if ENABLE_LLM_GENERATION and groq_key_available
    else "Extractive grounded fallback until GROQ_API_KEY is added",
)


Groq API key available: True
Generation mode: Groq Llama (llama-3.1-8b-instant)


## Step 3 — Download and Inspect the Bitext Dataset

The project uses the **Bitext Customer Support LLM Chatbot Training Dataset**. The important fields are:

- `instruction`: the customer question that enters the streaming pipeline.
- `category`: broad support area.
- `intent`: specific customer intent.
- `response`: reference response retained for analysis, but not injected into the RAG prompt.
- `flags`: optional linguistic annotations.

The pipeline samples records and adds operational fields such as ticket ID, customer ID, channel, priority, status, and event timestamp.

In [5]:
# =====================================================================
# DATA SOURCE — BITEXT CUSTOMER-SUPPORT DATASET
# =====================================================================

hf_dataset = load_dataset(DATASET_ID, split="train")
df_source = hf_dataset.to_pandas()

print("=" * 70)
print("  SOURCE DATASET PROFILE")
print("=" * 70)
print(f"Rows available : {len(df_source):,}")
print(f"Columns        : {list(df_source.columns)}")
print("\nFirst three source records:")
display(df_source.head(3))

required_source_columns = {"instruction", "category", "intent", "response"}
missing_source_columns = required_source_columns - set(df_source.columns)
if missing_source_columns:
    raise ValueError(f"Dataset schema changed. Missing columns: {sorted(missing_source_columns)}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

  SOURCE DATASET PROFILE
Rows available : 26,872
Columns        : ['flags', 'instruction', 'category', 'intent', 'response']

First three source records:


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


## Step 4 — Prepare Production-Simulated Ticket Events

A historical dataset does not arrive in real time by itself. This cell transforms each sampled row into an event that looks like a live support ticket.

A few controlled quality problems are intentionally injected to demonstrate:

- contract rejection,
- quarantine routing,
- duplicate removal,
- invalid-domain detection.

These errors are useful during the final presentation because they visibly prove that the governance layer works.

In [6]:
# =====================================================================
# EVENT ENRICHMENT — CREATE OPERATIONAL CUSTOMER-SUPPORT TICKETS
# =====================================================================

sample_n = min(SAMPLE_SIZE, len(df_source))
df_tickets = df_source.sample(n=sample_n, random_state=SEED).reset_index(drop=True).copy()

channels = ["chat", "email", "mobile_app", "web_form"]
priorities = ["low", "medium", "high"]
statuses = ["open", "open", "open", "pending"]
base_time = datetime.now(timezone.utc) - timedelta(minutes=sample_n)

# Rename the original fields into project terminology.
df_tickets = df_tickets.rename(columns={
    "instruction": "question",
    "response": "reference_response",
})

df_tickets["ticket_id"] = [f"T-{i:05d}" for i in range(1, len(df_tickets) + 1)]
df_tickets["customer_id"] = [f"C-{random.randint(1000, 9999)}" for _ in range(len(df_tickets))]
df_tickets["channel"] = [random.choice(channels) for _ in range(len(df_tickets))]
df_tickets["priority"] = [random.choice(priorities) for _ in range(len(df_tickets))]
df_tickets["status"] = [random.choice(statuses) for _ in range(len(df_tickets))]
df_tickets["event_timestamp"] = [
    (base_time + timedelta(minutes=i)).isoformat() for i in range(len(df_tickets))
]

# Normalise nullable source fields before they become streaming events.
for source_column in ["question", "category", "intent", "reference_response", "flags"]:
    if source_column not in df_tickets.columns:
        df_tickets[source_column] = ""
    df_tickets[source_column] = df_tickets[source_column].fillna("").astype(str)

# Reorder fields into the event schema used by the pipeline.
event_columns = [
    "ticket_id", "customer_id", "question", "category", "intent",
    "channel", "priority", "status", "event_timestamp",
    "reference_response", "flags",
]
df_tickets = df_tickets[event_columns]

# ---------------------------------------------------------------------
# INTENTIONAL DATA-QUALITY ERRORS FOR THE DEMO
# ---------------------------------------------------------------------
if len(df_tickets) >= 8:
    df_tickets.loc[0, "ticket_id"] = ""                 # Missing primary key
    df_tickets.loc[1, "question"] = ""                  # Empty customer question
    df_tickets.loc[2, "channel"] = "fax"                # Invalid channel domain
    df_tickets.loc[3, "event_timestamp"] = "bad-date"   # Invalid timestamp
    df_tickets.loc[4, "ticket_id"] = df_tickets.loc[5, "ticket_id"]  # Duplicate

raw_csv_path = PATHS["raw"] / "bitext_support_tickets_enriched.csv"
df_tickets.to_csv(raw_csv_path, index=False)

print("=" * 70)
print("  ENRICHED EVENT DATA")
print("=" * 70)
print(f"Rows prepared : {len(df_tickets):,}")
print(f"Saved to      : {raw_csv_path}")
print("\nSample event:")
print(json.dumps(df_tickets.iloc[6].to_dict(), indent=2, ensure_ascii=False))

  ENRICHED EVENT DATA
Rows prepared : 200
Saved to      : /content/supportai_final_project/data/raw/bitext_support_tickets_enriched.csv

Sample event:
{
  "ticket_id": "T-00007",
  "customer_id": "C-2679",
  "question": "I'd like to see the withdrwaal fee how can i do it",
  "category": "CANCEL",
  "intent": "check_cancellation_fee",
  "channel": "chat",
  "priority": "high",
  "status": "open",
  "event_timestamp": "2026-07-02T17:43:43.350179+00:00",
  "reference_response": "I'll do my best! To view the withdrawal fee, you can log in to your account and navigate to the \"Fee Schedule\" or \"Account Charges\" section. This will provide you with the details of the withdrawal fee and any associated charges.",
  "flags": "BCILPQZ"
}


## Step 5 — Raw Data Profiling

Before applying a contract, production engineers profile the incoming data to understand its condition. The profile checks nulls, duplicates, category distribution, and text length.

In [7]:
# =====================================================================
# RAW DATA QUALITY PROFILE
# =====================================================================

print("=" * 70)
print("  RAW TICKET DATA QUALITY PROFILE")
print("=" * 70)
print(f"Shape                     : {df_tickets.shape}")
print(f"Missing ticket IDs        : {(df_tickets['ticket_id'].astype(str).str.strip() == '').sum():,}")
print(f"Missing questions         : {(df_tickets['question'].astype(str).str.strip() == '').sum():,}")
print(f"Duplicate ticket IDs      : {df_tickets['ticket_id'].duplicated().sum():,}")
print(f"Unique categories         : {df_tickets['category'].nunique():,}")
print(f"Unique intents            : {df_tickets['intent'].nunique():,}")
print(f"Average question length   : {df_tickets['question'].astype(str).str.len().mean():.1f} characters")
print("\nTop categories:")
print(df_tickets["category"].value_counts().head(10).to_string())
print("=" * 70)

  RAW TICKET DATA QUALITY PROFILE
Shape                     : (200, 11)
Missing ticket IDs        : 1
Missing questions         : 1
Duplicate ticket IDs      : 1
Unique categories         : 11
Unique intents            : 27
Average question length   : 46.3 characters

Top categories:
category
ACCOUNT         36
CONTACT         30
INVOICE         24
REFUND          19
ORDER           18
PAYMENT         16
SHIPPING        15
DELIVERY        14
FEEDBACK        12
SUBSCRIPTION     9


## Step 6 — Data Contract with Pydantic v2

The contract formalizes the producer-consumer agreement. An event is accepted only when it satisfies the schema and business rules.

Critical checks include:

- non-empty ticket ID and customer question,
- valid channel, priority, and status domains,
- parseable UTC timestamp,
- minimum question length,
- no unexpected fields.

In [8]:
# =====================================================================
# DATA CONTRACT — MACHINE-ENFORCEABLE EVENT SCHEMA
# =====================================================================

ALLOWED_CHANNELS = {"chat", "email", "mobile_app", "web_form"}
ALLOWED_PRIORITIES = {"low", "medium", "high"}
ALLOWED_STATUSES = {"open", "pending", "closed", "escalated"}

class SupportTicketContract(BaseModel):
    """Schema and business rules for every incoming support-ticket event."""

    model_config = ConfigDict(strict=False, extra="forbid")

    ticket_id: str
    customer_id: str
    question: str = Field(min_length=5, max_length=2000)
    category: str
    intent: str
    channel: str
    priority: str
    status: str
    event_timestamp: datetime
    reference_response: str = ""
    flags: str = ""

    @field_validator("ticket_id", "customer_id", "category", "intent")
    @classmethod
    def required_string(cls, value: str) -> str:
        value = str(value).strip()
        if not value or value.lower() in {"nan", "none"}:
            raise ValueError("required field cannot be empty")
        return value

    @field_validator("question")
    @classmethod
    def validate_question(cls, value: str) -> str:
        value = re.sub(r"\s+", " ", str(value)).strip()
        if len(value) < 5:
            raise ValueError("question must contain at least 5 characters")
        return value

    @field_validator("channel")
    @classmethod
    def validate_channel(cls, value: str) -> str:
        value = str(value).strip().lower()
        if value not in ALLOWED_CHANNELS:
            raise ValueError(f"channel must be one of {sorted(ALLOWED_CHANNELS)}")
        return value

    @field_validator("priority")
    @classmethod
    def validate_priority(cls, value: str) -> str:
        value = str(value).strip().lower()
        if value not in ALLOWED_PRIORITIES:
            raise ValueError(f"priority must be one of {sorted(ALLOWED_PRIORITIES)}")
        return value

    @field_validator("status")
    @classmethod
    def validate_status(cls, value: str) -> str:
        value = str(value).strip().lower()
        if value not in ALLOWED_STATUSES:
            raise ValueError(f"status must be one of {sorted(ALLOWED_STATUSES)}")
        return value

print("✅ SupportTicketContract defined successfully.")

✅ SupportTicketContract defined successfully.


## Step 7 — Kafka-Style Broker, Producer, and Consumer

The course streaming lab uses a local broker simulation to demonstrate Kafka concepts without requiring a separate server. This implementation preserves the important Kafka ideas:

- named topic,
- multiple partitions,
- ordered offsets within each partition,
- asynchronous producer and consumers,
- key-based partition assignment,
- consumer-side validation.

Every event is first retained in the Bronze landing list. Valid events continue downstream, while rejected events are written to quarantine with the reason.

In [9]:
# =====================================================================
# STREAMING BACKBONE — MOCK KAFKA WITH PARTITIONS AND OFFSETS
# =====================================================================

class MockKafkaBroker:
    """
    Production-simulated Kafka broker used by the course-style Colab lab.

    Each topic contains ordered partition queues. The message key determines
    the partition, and every partition maintains its own monotonically
    increasing offset.
    """

    def __init__(self, partitions_per_topic: int = 3):
        self.partitions_per_topic = partitions_per_topic
        self.topics: dict[str, list[asyncio.Queue]] = {}
        self.offsets: dict[tuple[str, int], int] = defaultdict(int)

    def create_topic(self, topic: str) -> None:
        if topic not in self.topics:
            self.topics[topic] = [
                asyncio.Queue() for _ in range(self.partitions_per_topic)
            ]
            print(f"📡 Topic created: {topic} ({self.partitions_per_topic} partitions)")

    async def publish(self, topic: str, key: str, value: Optional[dict]) -> dict:
        if topic not in self.topics:
            self.create_topic(topic)

        partition = abs(hash(key)) % self.partitions_per_topic if value is not None else int(key)
        offset_key = (topic, partition)
        offset = self.offsets[offset_key]
        self.offsets[offset_key] += 1

        message = {
            "topic": topic,
            "partition": partition,
            "offset": offset,
            "key": key,
            "value": value,
            "broker_timestamp": datetime.now(timezone.utc).isoformat(),
        }
        await self.topics[topic][partition].put(message)
        return message

    async def consume(self, topic: str, partition: int) -> dict:
        return await self.topics[topic][partition].get()


async def ticket_producer(
    broker: MockKafkaBroker,
    records: list[dict],
    topic: str = "customer_support_tickets_raw",
) -> None:
    """Publishes historical Bitext rows one by one as live ticket events."""

    print(f"\n🚀 Producer starting — {len(records)} ticket events")
    for index, record in enumerate(records, start=1):
        key = str(record.get("ticket_id") or f"missing-{index}")
        message = await broker.publish(topic, key, record)
        if index <= 5 or index % 50 == 0:
            print(
                f"  PRODUCED {index:>3}/{len(records)} | "
                f"partition={message['partition']} offset={message['offset']} "
                f"ticket={key}"
            )
        await asyncio.sleep(STREAM_DELAY_SECONDS)

    # One termination marker per partition.
    for partition in range(broker.partitions_per_topic):
        await broker.publish(topic, str(partition), None)
    print("✅ Producer completed")


async def ticket_partition_consumer(
    broker: MockKafkaBroker,
    partition: int,
    bronze_events: list[dict],
    valid_events: list[dict],
    rejected_events: list[dict],
    topic: str = "customer_support_tickets_raw",
) -> None:
    """Consumes one Kafka partition, validates events, and routes failures."""

    while True:
        message = await broker.consume(topic, partition)
        record = message["value"]
        if record is None:
            break

        # Bronze captures the event as received, plus broker metadata.
        bronze_record = dict(record)
        bronze_record.update({
            "kafka_partition": message["partition"],
            "kafka_offset": message["offset"],
            "ingested_at": message["broker_timestamp"],
        })
        bronze_events.append(bronze_record)

        try:
            validated = SupportTicketContract(**record).model_dump(mode="json")
            validated.update({
                "kafka_partition": message["partition"],
                "kafka_offset": message["offset"],
                "ingested_at": message["broker_timestamp"],
            })
            valid_events.append(validated)
        except Exception as exc:
            rejected = bronze_record.copy()
            rejected["rejection_reason"] = str(exc)
            rejected_events.append(rejected)

    print(f"  ✅ Consumer partition {partition} stopped")


async def run_streaming_ingestion(df: pd.DataFrame):
    broker = MockKafkaBroker(partitions_per_topic=NUM_KAFKA_PARTITIONS)
    broker.create_topic("customer_support_tickets_raw")

    bronze_events, valid_events, rejected_events = [], [], []

    consumers = [
        asyncio.create_task(
            ticket_partition_consumer(
                broker, partition, bronze_events, valid_events, rejected_events
            )
        )
        for partition in range(NUM_KAFKA_PARTITIONS)
    ]

    producer = asyncio.create_task(
        ticket_producer(broker, df.to_dict(orient="records"))
    )

    await producer
    await asyncio.gather(*consumers)

    return (
        pd.DataFrame(bronze_events),
        pd.DataFrame(valid_events),
        pd.DataFrame(rejected_events),
    )

## Step 8 — Run Streaming Ingestion and Quarantine Invalid Events

This cell starts the producer and all partition consumers concurrently. It then saves:

- every received record to the Bronze landing CSV,
- contract-valid records to a staging file,
- rejected records and reasons to the quarantine zone.

In [10]:
# =====================================================================
# EXECUTE STREAMING INGESTION
# =====================================================================

try:
    # Colab/Jupyter supports top-level await in a code cell.
    df_bronze_stream, df_contract_valid, df_contract_rejected = await run_streaming_ingestion(df_tickets)
except RuntimeError:
    # Fallback for environments without top-level await support.
    df_bronze_stream, df_contract_valid, df_contract_rejected = asyncio.run(
        run_streaming_ingestion(df_tickets)
    )

bronze_csv = PATHS["bronze"] / "customer_support_tickets_raw.csv"
valid_csv = PATHS["silver"] / "contract_valid_staging.csv"
quarantine_csv = PATHS["quarantine"] / "contract_violations.csv"

df_bronze_stream.to_csv(bronze_csv, index=False)
df_contract_valid.to_csv(valid_csv, index=False)
if not df_contract_rejected.empty:
    df_contract_rejected.to_csv(quarantine_csv, index=False)

print("\n" + "=" * 70)
print("  STREAMING INGESTION RESULTS")
print("=" * 70)
print(f"Bronze events received : {len(df_bronze_stream):>6,}")
print(f"Contract-valid events  : {len(df_contract_valid):>6,}")
print(f"Rejected events        : {len(df_contract_rejected):>6,}")
print(f"Bronze landing file    : {bronze_csv}")
print(f"Quarantine file        : {quarantine_csv if not df_contract_rejected.empty else 'No rejected rows'}")
print("=" * 70)

if not df_contract_rejected.empty:
    print("\nRejected-event examples:")
    display(df_contract_rejected[["ticket_id", "question", "channel", "rejection_reason"]].head())

📡 Topic created: customer_support_tickets_raw (3 partitions)

🚀 Producer starting — 200 ticket events
  PRODUCED   1/200 | partition=0 offset=0 ticket=missing-1
  PRODUCED   2/200 | partition=0 offset=1 ticket=T-00002
  PRODUCED   3/200 | partition=0 offset=2 ticket=T-00003
  PRODUCED   4/200 | partition=0 offset=3 ticket=T-00004
  PRODUCED   5/200 | partition=1 offset=0 ticket=T-00006
  PRODUCED  50/200 | partition=1 offset=18 ticket=T-00050
  PRODUCED 100/200 | partition=2 offset=30 ticket=T-00100
  PRODUCED 150/200 | partition=0 offset=43 ticket=T-00150
  PRODUCED 200/200 | partition=2 offset=68 ticket=T-00200
✅ Producer completed
  ✅ Consumer partition 0 stopped
  ✅ Consumer partition 1 stopped
  ✅ Consumer partition 2 stopped

  STREAMING INGESTION RESULTS
Bronze events received :    200
Contract-valid events  :    196
Rejected events        :      4
Bronze landing file    : /content/supportai_final_project/data/bronze/customer_support_tickets_raw.csv
Quarantine file        : /con

,ticket_id,question,channel,rejection_reason
0,,I can't talk with a human agent,web_form,1 validation error for SupportTicketContract\n...
1,T-00002,,chat,1 validation error for SupportTicketContract\n...
2,T-00003,"I cannot pay, help me to inform of a problem w...",fax,1 validation error for SupportTicketContract\n...
3,T-00004,I want help speaking to customer service,chat,1 validation error for SupportTicketContract\n...


## Step 9 — Great Expectations Quality Suite

The contract validates each individual event. Great Expectations evaluates the accepted batch as a whole.

The suite checks:

1. `ticket_id` is never null.
2. `ticket_id` is unique.
3. `question` is never null.
4. `channel` belongs to the approved domain.
5. `priority` belongs to the approved domain.
6. question length remains within the business range.

The batch is cleaned and deduplicated before the final suite so that the pipeline can demonstrate both detection and correction.

In [11]:
# =====================================================================
# SILVER PREPARATION — CLEAN, NORMALISE, MASK, DEDUPLICATE
# =====================================================================

EMAIL_PATTERN = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

df_silver_pd = df_contract_valid.copy()

for column in ["ticket_id", "customer_id", "category", "intent", "channel", "priority", "status"]:
    df_silver_pd[column] = df_silver_pd[column].astype(str).str.strip()

df_silver_pd["channel"] = df_silver_pd["channel"].str.lower()
df_silver_pd["priority"] = df_silver_pd["priority"].str.lower()
df_silver_pd["status"] = df_silver_pd["status"].str.lower()
df_silver_pd["question"] = (
    df_silver_pd["question"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(EMAIL_PATTERN, "[EMAIL_REDACTED]", regex=True)
)

duplicate_count_before = df_silver_pd["ticket_id"].duplicated().sum()
df_silver_pd = df_silver_pd.drop_duplicates(subset=["ticket_id"], keep="last").reset_index(drop=True)

print(f"Duplicate ticket IDs removed: {duplicate_count_before}")
print(f"Silver candidate rows        : {len(df_silver_pd):,}")

Duplicate ticket IDs removed: 1
Silver candidate rows        : 195


In [12]:
# =====================================================================
# GREAT EXPECTATIONS — FORMAL EXPECTATION SUITE
# =====================================================================

import great_expectations as gx

try:
    ge_tickets = gx.from_pandas(df_silver_pd.copy())

    ge_tickets.expect_column_values_to_not_be_null("ticket_id")
    ge_tickets.expect_column_values_to_be_unique("ticket_id")
    ge_tickets.expect_column_values_to_not_be_null("question")
    ge_tickets.expect_column_values_to_be_in_set("channel", sorted(ALLOWED_CHANNELS))
    ge_tickets.expect_column_values_to_be_in_set("priority", sorted(ALLOWED_PRIORITIES))
    ge_tickets.expect_column_value_lengths_to_be_between(
        "question", min_value=5, max_value=2000
    )

    ge_validation = ge_tickets.validate(result_format="SUMMARY")
    try:
        ge_report = ge_validation.to_json_dict()
    except AttributeError:
        ge_report = dict(ge_validation)

    ge_passed = bool(ge_report.get("success", False))

except Exception as exc:
    # Version-safe fallback keeps the demo running while preserving the same rules.
    logger.warning(f"Great Expectations API fallback activated: {exc}")
    checks = {
        "ticket_id_not_null": df_silver_pd["ticket_id"].notna().all(),
        "ticket_id_unique": df_silver_pd["ticket_id"].is_unique,
        "question_not_null": df_silver_pd["question"].notna().all(),
        "channel_domain": df_silver_pd["channel"].isin(ALLOWED_CHANNELS).all(),
        "priority_domain": df_silver_pd["priority"].isin(ALLOWED_PRIORITIES).all(),
        "question_length": df_silver_pd["question"].str.len().between(5, 2000).all(),
    }
    ge_passed = all(checks.values())
    ge_report = {
        "success": ge_passed,
        "fallback_checks": {k: bool(v) for k, v in checks.items()},
    }

ge_report_path = PATHS["quality"] / "great_expectations_validation.json"
with open(ge_report_path, "w", encoding="utf-8") as file:
    json.dump(ge_report, file, indent=2, ensure_ascii=False, default=str)

print("=" * 70)
print("  GREAT EXPECTATIONS VALIDATION")
print("=" * 70)
print(f"Overall verdict : {'PASSED ✅' if ge_passed else 'FAILED ❌'}")
print(f"Rows evaluated  : {len(df_silver_pd):,}")
print(f"Report path     : {ge_report_path}")
print("=" * 70)

# PRODUCTION QUALITY GATE: a failure stops this notebook/pipeline immediately.
if not ge_passed:
    failed_batch_path = PATHS["quarantine"] / "quality_gate_failed_batch.csv"
    df_silver_pd.to_csv(failed_batch_path, index=False)
    raise RuntimeError(
        f"Great Expectations quality gate failed. Downstream stages are blocked. "
        f"Rejected batch: {failed_batch_path}"
    )
print("✅ Quality gate passed — downstream stages are allowed to run.")


  return datetime.utcnow().replace(tzinfo=utc)



  GREAT EXPECTATIONS VALIDATION
Overall verdict : PASSED ✅
Rows evaluated  : 195
Report path     : /content/supportai_final_project/quality_reports/great_expectations_validation.json
✅ Quality gate passed — downstream stages are allowed to run.


## Step 10 — DAMA Six-Dimension Quality Engine

To follow the Day 4 lab style, this additional engine summarizes the same batch across the six DAMA dimensions:

- Completeness
- Accuracy
- Consistency
- Timeliness
- Uniqueness
- Validity

The overall quality gate passes only when all critical dimensions pass.

In [13]:
# =====================================================================
# DAMA SIX-DIMENSION QUALITY ENGINE
# =====================================================================

class SupportTicketQualityEngine:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.results: dict[str, dict] = {}

    def _record(self, name: str, passed: bool, detail: str) -> None:
        self.results[name] = {
            "passed": bool(passed),
            "status": "PASSED" if passed else "FAILED",
            "detail": detail,
        }
        logger.success(f"[PASSED] {name}: {detail}") if passed else logger.warning(
            f"[FAILED] {name}: {detail}"
        )

    def run(self) -> dict:
        # 1. Completeness
        critical_nulls = self.df[["ticket_id", "question", "category", "intent"]].isna().sum().sum()
        self._record("1_completeness", critical_nulls == 0,
                     f"{critical_nulls:,} null values in critical fields")

        # 2. Accuracy: operational text length range
        invalid_lengths = (~self.df["question"].str.len().between(5, 2000)).sum()
        self._record("2_accuracy", invalid_lengths == 0,
                     f"{invalid_lengths:,} questions outside the 5–2000 character range")

        # 3. Consistency: domains are standardised
        invalid_domains = (
            (~self.df["channel"].isin(ALLOWED_CHANNELS)).sum()
            + (~self.df["priority"].isin(ALLOWED_PRIORITIES)).sum()
            + (~self.df["status"].isin(ALLOWED_STATUSES)).sum()
        )
        self._record("3_consistency", invalid_domains == 0,
                     f"{invalid_domains:,} non-standard domain values")

        # 4. Timeliness: timestamps are parseable and not in the future
        timestamps = pd.to_datetime(self.df["event_timestamp"], errors="coerce", utc=True)
        invalid_time = timestamps.isna().sum() + (timestamps > pd.Timestamp.now(tz="UTC") + pd.Timedelta(minutes=5)).sum()
        self._record("4_timeliness", invalid_time == 0,
                     f"{invalid_time:,} invalid or future timestamps")

        # 5. Uniqueness
        duplicates = self.df["ticket_id"].duplicated().sum()
        self._record("5_uniqueness", duplicates == 0,
                     f"{duplicates:,} duplicate ticket IDs")

        # 6. Validity: ID formats
        invalid_ticket_ids = (~self.df["ticket_id"].str.match(r"^T-\d{5}$", na=False)).sum()
        invalid_customer_ids = (~self.df["customer_id"].str.match(r"^C-\d{4}$", na=False)).sum()
        invalid_ids = invalid_ticket_ids + invalid_customer_ids
        self._record("6_validity", invalid_ids == 0,
                     f"{invalid_ids:,} IDs violating the declared format")

        return {
            "passed": all(v["passed"] for v in self.results.values()),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "row_count": len(self.df),
            "dimensions": self.results,
        }


dq_engine = SupportTicketQualityEngine(df_silver_pd)
dq_report = dq_engine.run()

dq_report_path = PATHS["quality"] / "dama_quality_report.json"
with open(dq_report_path, "w", encoding="utf-8") as file:
    json.dump(dq_report, file, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("  DAMA DATA QUALITY REPORT")
print("=" * 70)
for dimension, result in dq_report["dimensions"].items():
    icon = "✅" if result["passed"] else "❌"
    print(f"{icon} {dimension:<22} | {result['detail']}")
print("-" * 70)
print(f"Overall quality gate: {'PASSED ✅' if dq_report['passed'] else 'FAILED ❌'}")
print("=" * 70)

if not (ge_passed and dq_report["passed"]):
    raise RuntimeError("Quality gate failed. Review the reports before writing Silver/Gold data.")

  return datetime.utcnow().replace(tzinfo=utc)



20:58:25 | SUCCESS  | [PASSED] 1_completeness: 0 null values in critical fields
20:58:25 | SUCCESS  | [PASSED] 2_accuracy: 0 questions outside the 5–2000 character range
20:58:25 | SUCCESS  | [PASSED] 3_consistency: 0 non-standard domain values
20:58:25 | SUCCESS  | [PASSED] 4_timeliness: 0 invalid or future timestamps
20:58:25 | SUCCESS  | [PASSED] 5_uniqueness: 0 duplicate ticket IDs
20:58:25 | SUCCESS  | [PASSED] 6_validity: 0 IDs violating the declared format

  DAMA DATA QUALITY REPORT
✅ 1_completeness         | 0 null values in critical fields
✅ 2_accuracy             | 0 questions outside the 5–2000 character range
✅ 3_consistency          | 0 non-standard domain values
✅ 4_timeliness           | 0 invalid or future timestamps
✅ 5_uniqueness           | 0 duplicate ticket IDs
✅ 6_validity             | 0 IDs violating the declared format
----------------------------------------------------------------------
Overall quality gate: PASSED ✅


## Step 11 — Configure Spark and Delta Lake

This cell creates a Spark session with the Delta Lake extensions required for:

- ACID transactions,
- schema enforcement,
- transaction history,
- `MERGE`/UPSERT operations,
- Bronze, Silver, and Gold tables.

In [14]:
# =====================================================================
# SPARK + DELTA LAKE SESSION
# =====================================================================

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

try:
    SparkSession.getActiveSession().stop() if SparkSession.getActiveSession() else None
except Exception:
    pass

spark_builder = (
    SparkSession.builder
    .appName("SupportAI_Final_Project")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
)

spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Spark version: {spark.version}")
print("✅ Delta Lake session configured")

✅ Spark version: 4.0.1
✅ Delta Lake session configured


## Step 12 — Write the Bronze Delta Table

The Bronze table preserves events close to their original form and includes Kafka metadata. Schema enforcement is explicit: Spark will reject records that do not conform to the declared table structure.

In [15]:
# =====================================================================
# BRONZE LAYER — RAW STREAMING EVENTS
# =====================================================================

BRONZE_SCHEMA = StructType([
    StructField("ticket_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("question", StringType(), True),
    StructField("category", StringType(), True),
    StructField("intent", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("status", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("reference_response", StringType(), True),
    StructField("flags", StringType(), True),
    StructField("kafka_partition", IntegerType(), True),
    StructField("kafka_offset", IntegerType(), True),
    StructField("ingested_at", StringType(), True),
])

bronze_columns = [field.name for field in BRONZE_SCHEMA.fields]
df_bronze_for_spark = df_bronze_stream.reindex(columns=bronze_columns).copy()

for col_name in [c for c in bronze_columns if c not in {"kafka_partition", "kafka_offset"}]:
    df_bronze_for_spark[col_name] = df_bronze_for_spark[col_name].fillna("").astype(str)
for col_name in ["kafka_partition", "kafka_offset"]:
    df_bronze_for_spark[col_name] = pd.to_numeric(df_bronze_for_spark[col_name], errors="coerce").fillna(-1).astype(int)

bronze_spark_df = spark.createDataFrame(df_bronze_for_spark, schema=BRONZE_SCHEMA)
bronze_delta_path = str(PATHS["bronze"] / "tickets_delta")

(
    bronze_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(bronze_delta_path)
)

print("=" * 70)
print("  BRONZE DELTA TABLE")
print("=" * 70)
print(f"Rows written : {spark.read.format('delta').load(bronze_delta_path).count():,}")
print(f"Path         : {bronze_delta_path}")
print("Schema:")
spark.read.format("delta").load(bronze_delta_path).printSchema()

  return datetime.utcnow().replace(tzinfo=utc)



  BRONZE DELTA TABLE
Rows written : 200
Path         : /content/supportai_final_project/data/bronze/tickets_delta
Schema:
root
 |-- ticket_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- question: string (nullable = true)
 |-- category: string (nullable = true)
 |-- intent: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- reference_response: string (nullable = true)
 |-- flags: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: integer (nullable = true)
 |-- ingested_at: string (nullable = true)



## Step 12B — Prove Delta Schema Enforcement

This controlled negative test attempts to append an unexpected column **without** `mergeSchema`. A production Delta table must reject the write. The caught exception is saved as rubric evidence.

In [16]:
# =====================================================================
# DELTA SCHEMA ENFORCEMENT — CONTROLLED FAILURE EVIDENCE
# =====================================================================

schema_enforcement_evidence = {
    "attempted_change": "append unexpected_new_column",
    "blocked": False,
    "passed": False,
}

try:
    incompatible_df = bronze_spark_df.limit(1).withColumn(
        "unexpected_new_column", F.lit("this_schema_drift_must_be_rejected")
    )
    (
        incompatible_df.write
        .format("delta")
        .mode("append")
        .save(bronze_delta_path)
    )
except Exception as exc:
    schema_enforcement_evidence.update({
        "blocked": True,
        "passed": True,
        "error": str(exc)[:4000],
    })
    print("✅ Delta schema enforcement correctly blocked the incompatible append.")
else:
    schema_enforcement_evidence["error"] = "Delta unexpectedly accepted schema drift."
    raise RuntimeError(schema_enforcement_evidence["error"])

schema_evidence_path = PATHS["quality"] / "delta_schema_enforcement_evidence.json"
schema_evidence_path.write_text(
    json.dumps(schema_enforcement_evidence, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Evidence saved to: {schema_evidence_path}")


✅ Delta schema enforcement correctly blocked the incompatible append.
Evidence saved to: /content/supportai_final_project/quality_reports/delta_schema_enforcement_evidence.json


## Step 13 — Build the Silver Delta Table

Silver contains only contract-valid and quality-approved records. It applies:

- whitespace normalization,
- lowercase domain normalization,
- email masking,
- duplicate removal,
- explicit filtering of invalid records.

In [17]:
# =====================================================================
# SILVER LAYER — CLEAN, VALIDATED, DEDUPLICATED TICKETS
# =====================================================================

silver_source_df = spark.createDataFrame(df_silver_pd.astype(str))

silver_spark_df = (
    silver_source_df
    .withColumn("question", F.trim(F.regexp_replace(F.col("question"), r"\s+", " ")))
    .withColumn("question", F.regexp_replace(
        F.col("question"),
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        "[EMAIL_REDACTED]",
    ))
    .withColumn("channel", F.lower(F.trim(F.col("channel"))))
    .withColumn("priority", F.lower(F.trim(F.col("priority"))))
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    .withColumn("silver_processed_at", F.current_timestamp())
    .dropDuplicates(["ticket_id"])
)

silver_delta_path = str(PATHS["silver"] / "tickets_delta")
(
    silver_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_delta_path)
)

silver_check = spark.read.format("delta").load(silver_delta_path)
print("=" * 70)
print("  SILVER DELTA TABLE")
print("=" * 70)
print(f"Rows written : {silver_check.count():,}")
print(f"Path         : {silver_delta_path}")
silver_check.select("ticket_id", "question", "category", "intent", "channel", "status").show(5, truncate=70)

  return datetime.utcnow().replace(tzinfo=utc)



  SILVER DELTA TABLE
Rows written : 195
Path         : /content/supportai_final_project/data/silver/tickets_delta
+---------+-------------------------------------------------------+------------+-----------------------+----------+------+
|ticket_id|                                               question|    category|                 intent|   channel|status|
+---------+-------------------------------------------------------+------------+-----------------------+----------+------+
|  T-00006|            where to sign up to the company nmewsletter|SUBSCRIPTION|newsletter_subscription|mobile_app|  open|
|  T-00007|     I'd like to see the withdrwaal fee how can i do it|      CANCEL| check_cancellation_fee|      chat|  open|
|  T-00008|                           I want to speak with someone|     CONTACT|    contact_human_agent|     email|  open|
|  T-00009|                   can you help me getting bill #85632?|     INVOICE|            get_invoice|     email|  open|
|  T-00010|I don't know h

## Step 14 — Build Gold Business Tables

Gold tables are ready for analytics and AI serving. This project creates:

1. `support_intent_summary`: counts by category and intent.
2. `support_channel_summary`: counts by channel and priority.
3. the clean ticket table used by downstream customer-support applications.

In [18]:
# =====================================================================
# GOLD LAYER — BUSINESS-READY AGGREGATIONS
# =====================================================================

silver_df = spark.read.format("delta").load(silver_delta_path)

gold_intent_df = (
    silver_df.groupBy("category", "intent")
    .agg(
        F.count("*").alias("ticket_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
    )
    .orderBy(F.desc("ticket_count"))
)

gold_channel_df = (
    silver_df.groupBy("channel", "priority")
    .agg(F.count("*").alias("ticket_count"))
    .orderBy(F.desc("ticket_count"))
)

gold_intent_path = str(PATHS["gold"] / "support_intent_summary_delta")
gold_channel_path = str(PATHS["gold"] / "support_channel_summary_delta")

gold_intent_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_intent_path)
gold_channel_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_channel_path)

print("=" * 70)
print("  GOLD BUSINESS TABLES")
print("=" * 70)
print("Top category/intent combinations:")
gold_intent_df.show(10, truncate=45)
print("Channel and priority summary:")
gold_channel_df.show(10, truncate=False)

  GOLD BUSINESS TABLES
Top category/intent combinations:
+------------+------------------------+------------+----------------+
|    category|                  intent|ticket_count|unique_customers|
+------------+------------------------+------------+----------------+
|     CONTACT|     contact_human_agent|          19|              19|
|     INVOICE|             get_invoice|          14|              14|
|     ACCOUNT|          create_account|          11|              11|
|    DELIVERY|        delivery_options|          10|              10|
|     INVOICE|           check_invoice|           9|               9|
|SUBSCRIPTION| newsletter_subscription|           9|               9|
|     ACCOUNT|            edit_account|           9|               9|
|     CONTACT|contact_customer_service|           9|               9|
|    SHIPPING| change_shipping_address|           8|               8|
|     ACCOUNT|          delete_account|           8|               8|
+------------+-------------------

## Step 15 — Demonstrate Delta `MERGE` / UPSERT

A customer-support ticket changes over time. Instead of inserting duplicate copies, Delta `MERGE` updates matching records and inserts unseen records.

The demonstration changes a few open tickets to `closed` and adds a processing timestamp.

In [19]:
# =====================================================================
# DELTA MERGE / UPSERT DEMONSTRATION
# =====================================================================

merge_candidates = (
    silver_df.select("ticket_id")
    .limit(3)
    .toPandas()["ticket_id"]
    .tolist()
)

updates_pd = pd.DataFrame({
    "ticket_id": merge_candidates,
    "new_status": ["closed"] * len(merge_candidates),
    "resolution_note": ["Resolved during final-project demonstration"] * len(merge_candidates),
})
updates_spark = spark.createDataFrame(updates_pd)

silver_delta_table = DeltaTable.forPath(spark, silver_delta_path)
(
    silver_delta_table.alias("target")
    .merge(updates_spark.alias("source"), "target.ticket_id = source.ticket_id")
    .whenMatchedUpdate(set={
        "status": "source.new_status",
    })
    .execute()
)

print("✅ MERGE completed. Updated records:")
spark.read.format("delta").load(silver_delta_path).where(
    F.col("ticket_id").isin(merge_candidates)
).select("ticket_id", "status").show(truncate=False)

print("Delta transaction history:")
silver_delta_table.history(5).select("version", "timestamp", "operation").show(truncate=False)

✅ MERGE completed. Updated records:
+---------+------+
|ticket_id|status|
+---------+------+
|T-00008  |closed|
|T-00006  |closed|
|T-00007  |closed|
+---------+------+

Delta transaction history:
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-07-02 20:58:59.91 |MERGE    |
|0      |2026-07-02 20:58:51.995|WRITE    |
+-------+-----------------------+---------+



## Step 16 — Create the NovaStore Knowledge Base

The RAG system must answer from approved company policies, not from the Bitext reference answer. This notebook creates eight fictional NovaStore policy documents covering:

- refunds and returns,
- shipping and delivery,
- cancellation,
- payments and billing,
- account access,
- damaged or missing items,
- order tracking,
- human escalation.

Each document contains metadata required for citations and governance.

In [20]:
# =====================================================================
# GOVERNED KNOWLEDGE BASE — NOVASTORE POLICY DOCUMENTS
# =====================================================================

KNOWLEDGE_DOCUMENTS = [{'doc_id': 'KB-ACC-001', 'title': 'Account Access and Password Recovery', 'category': 'account_password', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'account_and_password.md', 'text': '# NovaStore Account Access and Password Recovery\n\n## Password reset\nCustomers can reset a password by selecting **Forgot Password** on the sign-in page. A verification link is sent to the registered email address. The link expires after **15 minutes** and can be used only once.\n\n## Verification email not received\nCustomers should:\n1. Check spam or junk folders.\n2. Confirm that the email address was entered correctly.\n3. Wait up to **5 minutes**.\n4. Request one additional reset email.\n\nRepeated requests may invalidate earlier links.\n\n## Locked accounts\nAn account may be temporarily locked after several unsuccessful sign-in attempts. The lock is normally removed after **30 minutes**. Customers should avoid repeated attempts during the lock period.\n\n## Email or phone changes\nChanges to registered contact details require identity verification. Support agents must not ask customers to provide full passwords, complete card numbers, or one-time verification codes.\n\n## Suspicious access\nIf unauthorized access is suspected, the customer should reset the password immediately, review recent orders, and contact support. The case should be escalated to the security team if an unknown order or account change is found.'}, {'doc_id': 'KB-DMG-001', 'title': 'Damaged, Incorrect, or Missing Items Policy', 'category': 'damaged_missing_items', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'damaged_or_missing_items.md', 'text': '# NovaStore Damaged, Incorrect, or Missing Items Policy\n\n## Reporting window\nCustomers should report a damaged, incorrect, or missing item within **48 hours** of delivery.\n\n## Required information\nThe customer should provide:\n- Order number\n- Description of the issue\n- Clear photos of the item and packaging\n- Photo of the shipping label when available\n\n## Damaged items\nAfter verification, NovaStore may offer a replacement, refund, or store credit. The customer should keep the item and packaging until support provides instructions.\n\n## Incorrect items\nIf the wrong product was delivered, NovaStore provides return instructions and covers reasonable return shipping. A replacement is sent after the issue is confirmed, subject to stock availability.\n\n## Missing items\nFor multi-item orders, customers should check whether the order was split into multiple shipments. If the package is marked delivered but is missing, the customer should check nearby locations and wait up to **24 hours** before reporting it.\n\n## Escalation\nCases involving high-value items, repeated delivery claims, or suspected package tampering must be escalated for manual review.'}, {'doc_id': 'KB-CAN-001', 'title': 'Order Cancellation Policy', 'category': 'order_cancellation', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'order_cancellation.md', 'text': '# NovaStore Order Cancellation Policy\n\n## When an order can be cancelled\nAn order can be cancelled while its status is **Pending** or **Processing**. Cancellation is no longer guaranteed after the order status changes to **Packed** or **Shipped**.\n\n## How to cancel\nCustomers can request cancellation from the My Orders page or by contacting customer support with the order number. A cancellation request is not complete until the customer receives a confirmation message.\n\n## Payment reversal\nFor successfully cancelled orders:\n- Card payments are reversed within **3–7 business days**.\n- Digital-wallet payments are usually reversed within **1–3 business days**.\n- Bank processing times may vary.\n\n## Partially shipped orders\nIf only part of an order has shipped, the unshipped items may still be cancelled. Shipped items must follow the standard return process after delivery.\n\n## Failed cancellation\nIf cancellation is not possible because the order has already shipped, the customer may refuse delivery where permitted or request a return after receiving the order.\n\n## Escalation\nA human agent should review cases involving duplicate orders, suspected fraud, or cancellation requests submitted before shipment but not processed correctly.'}, {'doc_id': 'KB-TRK-001', 'title': 'Order Tracking and Status Guide', 'category': 'order_tracking', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'order_tracking.md', 'text': '# NovaStore Order Tracking and Status Guide\n\n## Order statuses\n- **Pending:** Payment or order verification is incomplete.\n- **Processing:** Payment is confirmed and the order is being prepared.\n- **Packed:** The package is ready for carrier pickup.\n- **Shipped:** The carrier has received the package.\n- **Out for Delivery:** Delivery is expected that day.\n- **Delivered:** The carrier recorded delivery.\n- **Cancelled:** The order will not be shipped.\n- **Returned:** The package was received back by NovaStore.\n\n## Tracking information\nTracking information is usually available within **24 hours** after the status changes to Shipped. A tracking number may show no movement until the carrier performs the first scan.\n\n## No tracking update\nIf tracking has not changed for **48 hours**, the customer may contact support. Support can open a carrier investigation if the shipment is beyond the estimated delivery period.\n\n## Split shipments\nItems from the same order may be shipped separately. Each shipment may have a different tracking number and delivery date.'}, {'doc_id': 'KB-PAY-001', 'title': 'Payments and Billing Policy', 'category': 'payments_billing', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'payments_and_billing.md', 'text': '# NovaStore Payments and Billing Policy\n\n## Accepted payment methods\nNovaStore accepts major debit and credit cards, supported digital wallets, and approved local payment methods displayed during checkout.\n\n## Payment authorization\nA temporary authorization hold may appear when an order is placed. The final charge is completed after the order is confirmed. If an order fails or is cancelled, the hold may remain visible for **3–7 business days**, depending on the bank.\n\n## Declined payments\nPayments may be declined because of:\n- Incorrect card information\n- Insufficient funds\n- Bank security restrictions\n- Expired cards\n- Billing-address mismatch\n\nCustomers should verify their details and contact their bank before retrying repeatedly.\n\n## Duplicate charges\nA pending authorization and a completed charge may look like duplicate charges. Customers should wait up to **48 hours** for pending entries to disappear. If two completed charges remain, support will investigate after receiving the order number, payment date, and last four digits of the payment method.\n\n## Invoices\nInvoices become available after payment confirmation and can be downloaded from the My Orders page. Invoice details cannot normally be changed after the invoice is issued unless required by law.'}, {'doc_id': 'KB-REF-001', 'title': 'Refunds and Returns Policy', 'category': 'refunds_returns', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'refunds_and_returns.md', 'text': '# NovaStore Refunds and Returns Policy\n\n## Eligibility\nCustomers may request a return within **14 calendar days** of receiving an order. The item must be unused, in its original condition, and returned with all included accessories and packaging. Proof of purchase or the order number is required.\n\n## Non-returnable items\nDigital products, personalized products, opened hygiene-sensitive items, and gift cards are not eligible for return unless they are defective or were delivered incorrectly.\n\n## Return process\n1. Submit a return request through the customer support portal.\n2. Provide the order number and the reason for the return.\n3. Wait for the return authorization and shipping instructions.\n4. Send the item using the approved return method.\n\n## Refund timing\nAfter the returned item is received and inspected, approved refunds are issued to the original payment method within **5–7 business days**. Bank processing may require an additional **2–5 business days**.\n\n## Return shipping\nIf the return is caused by a defective, damaged, or incorrect item, NovaStore covers the return shipping cost. For change-of-mind returns, the customer is responsible for return shipping.\n\n## Escalation\nContact a human support agent if the refund has not appeared within **10 business days** after approval.'}, {'doc_id': 'KB-SHP-001', 'title': 'Shipping and Delivery Policy', 'category': 'shipping_delivery', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'shipping_and_delivery.md', 'text': '# NovaStore Shipping and Delivery Policy\n\n## Processing time\nOrders are normally processed within **1–2 business days** after payment confirmation. Orders placed on weekends or public holidays are processed on the next business day.\n\n## Delivery estimates\n- Standard delivery: **3–7 business days**\n- Express delivery: **1–3 business days**\n- Remote-area delivery: may require an additional **2–4 business days**\n\nDelivery times are estimates and may be affected by weather, carrier delays, customs checks, or incorrect address details.\n\n## Shipping address changes\nCustomers may request an address change only before the order status changes to **Shipped**. Once the package has been handed to the carrier, the address cannot be changed by NovaStore.\n\n## Delayed delivery\nA package is considered delayed when it has not arrived **2 business days** after the estimated delivery date. Customers should first check the tracking page. If there is no movement for **48 hours**, customer support can open a carrier investigation.\n\n## Delivery confirmation\nThe carrier may mark the order as delivered before the package is physically received. Customers should allow up to **24 hours**, check nearby locations, and ask household members or building reception before reporting the package as missing.'}, {'doc_id': 'KB-ESC-001', 'title': 'Customer Support Escalation and Response Policy', 'category': 'support_escalation', 'version': '1.0', 'effective_date': '2026-07-01', 'source': 'support_escalation.md', 'text': '# NovaStore Customer Support Escalation and Response Policy\n\n## Automated support scope\nThe automated assistant may answer general questions about orders, shipping, returns, payments, account access, and published company policies.\n\n## When to escalate to a human agent\nEscalation is required when:\n- The customer reports fraud or unauthorized activity.\n- The request involves a high-value order.\n- The customer disputes a previous decision.\n- The issue cannot be resolved using the knowledge base.\n- The customer has contacted support more than twice about the same unresolved issue.\n- Personal identity verification is required.\n- The customer explicitly asks to speak with a human agent.\n\n## Response targets\n- General questions: immediate automated response\n- Standard agent cases: within **4 business hours**\n- Urgent security or fraud cases: within **1 business hour**\n- Carrier investigations: initial update within **2 business days**\n\n## Safety and privacy\nAgents and automated systems must not request full passwords, complete payment-card numbers, or one-time verification codes. Sensitive customer data must be masked in logs and excluded from the vector database.\n\n## Uncertain answers\nIf the retrieved information is incomplete, conflicting, or outdated, the assistant must state that it cannot confirm the answer and escalate instead of guessing.'}]

for document in KNOWLEDGE_DOCUMENTS:
    output_path = PATHS["knowledge_base"] / document["source"]
    output_path.write_text(document["text"], encoding="utf-8")

kb_metadata_df = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "version": d["version"],
        "effective_date": d["effective_date"],
        "source": d["source"],
    }
    for d in KNOWLEDGE_DOCUMENTS
])
kb_metadata_df.to_csv(PATHS["knowledge_base"] / "metadata.csv", index=False)

print("=" * 70)
print("  NOVASTORE KNOWLEDGE BASE")
print("=" * 70)
print(f"Documents created : {len(KNOWLEDGE_DOCUMENTS)}")
print(f"Directory         : {PATHS['knowledge_base']}")
display(kb_metadata_df)

  return datetime.utcnow().replace(tzinfo=utc)



  NOVASTORE KNOWLEDGE BASE
Documents created : 8
Directory         : /content/supportai_final_project/knowledge_base


,doc_id,title,category,version,effective_date,source
0,KB-ACC-001,Account Access and Password Recovery,account_password,1.0,2026-07-01,account_and_password.md
1,KB-DMG-001,"Damaged, Incorrect, or Missing Items Policy",damaged_missing_items,1.0,2026-07-01,damaged_or_missing_items.md
2,KB-CAN-001,Order Cancellation Policy,order_cancellation,1.0,2026-07-01,order_cancellation.md
3,KB-TRK-001,Order Tracking and Status Guide,order_tracking,1.0,2026-07-01,order_tracking.md
4,KB-PAY-001,Payments and Billing Policy,payments_billing,1.0,2026-07-01,payments_and_billing.md
5,KB-REF-001,Refunds and Returns Policy,refunds_returns,1.0,2026-07-01,refunds_and_returns.md
6,KB-SHP-001,Shipping and Delivery Policy,shipping_delivery,1.0,2026-07-01,shipping_and_delivery.md
7,KB-ESC-001,Customer Support Escalation and Response Policy,support_escalation,1.0,2026-07-01,support_escalation.md


## Step 17 — Semantic Chunking with Metadata

The chunker respects Markdown section boundaries first, then creates overlapping sentence windows when a section is long. Every chunk retains:

- document ID,
- title,
- category,
- section name,
- source filename,
- version and effective date.

This metadata allows the final answer to cite its evidence.

In [21]:
# =====================================================================
# RAG STAGE 1 — STRUCTURE-AWARE CHUNKING
# =====================================================================

def split_sentences(text: str) -> list[str]:
    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", text.strip())
        if sentence.strip()
    ]


def chunk_knowledge_base(
    documents: list[dict],
    max_sentences: int = 4,
    overlap_sentences: int = 1,
) -> list[dict]:
    """Splits documents by H2 headings and then by overlapping sentences."""

    chunks = []
    for document in documents:
        text = document["text"]
        parts = re.split(r"\n##\s+", text)

        # Ignore the H1 title section when no policy text follows it.
        for section_index, part in enumerate(parts):
            part = part.strip()
            if not part:
                continue

            lines = part.splitlines()
            section_title = lines[0].lstrip("# ").strip()
            section_body = " ".join(line.strip() for line in lines[1:] if line.strip())
            if not section_body:
                continue

            sentences = split_sentences(section_body)
            step = max(1, max_sentences - overlap_sentences)

            for start in range(0, len(sentences), step):
                window = sentences[start:start + max_sentences]
                if not window:
                    continue
                chunk_text = " ".join(window)
                chunk_id = f"{document['doc_id']}-S{section_index:02d}-C{start:03d}"
                chunks.append({
                    "id": chunk_id,
                    "text": chunk_text,
                    "doc_id": document["doc_id"],
                    "title": document["title"],
                    "category": document["category"],
                    "section": section_title,
                    "source": document["source"],
                    "version": document["version"],
                    "effective_date": document["effective_date"],
                })

    return chunks


KB_CHUNKS = chunk_knowledge_base(KNOWLEDGE_DOCUMENTS)
chunks_jsonl_path = PATHS["knowledge_base"] / "kb_chunks.jsonl"
with open(chunks_jsonl_path, "w", encoding="utf-8") as file:
    for chunk in KB_CHUNKS:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"📄 {len(KNOWLEDGE_DOCUMENTS)} documents → {len(KB_CHUNKS)} metadata-rich chunks")
print(f"Saved chunk file: {chunks_jsonl_path}")
print("\nExample chunk:")
print(json.dumps(KB_CHUNKS[0], indent=2, ensure_ascii=False))

📄 8 documents → 50 metadata-rich chunks
Saved chunk file: /content/supportai_final_project/knowledge_base/kb_chunks.jsonl

Example chunk:
{
  "id": "KB-ACC-001-S01-C000",
  "text": "Customers can reset a password by selecting **Forgot Password** on the sign-in page. A verification link is sent to the registered email address. The link expires after **15 minutes** and can be used only once.",
  "doc_id": "KB-ACC-001",
  "title": "Account Access and Password Recovery",
  "category": "account_password",
  "section": "Password reset",
  "source": "account_and_password.md",
  "version": "1.0",
  "effective_date": "2026-07-01"
}


## Step 18 — Build ChromaDB Vector Index and BM25 Keyword Index

The retrieval layer uses two complementary indexes:

- **Dense vector retrieval:** finds semantically similar passages even when wording differs.
- **BM25 sparse retrieval:** captures exact terms, IDs, time periods, and policy-specific language.

ChromaDB uses an HNSW-family approximate nearest-neighbor index internally.

In [22]:
# =====================================================================
# RAG STAGE 2 — VECTOR INDEX AND BM25 INDEX
# =====================================================================

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer


def tokenize_for_bm25(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9]+", text.lower())


class BM25Index:
    def __init__(self, chunks: list[dict]):
        self.chunks = chunks
        tokenized = [tokenize_for_bm25(chunk["text"]) for chunk in chunks]
        self.index = BM25Okapi(tokenized)

    def search(self, query: str, top_k: int = 10) -> list[dict]:
        scores = self.index.get_scores(tokenize_for_bm25(query))
        ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)
        results = []
        for idx, score in ranked[:top_k]:
            item = dict(self.chunks[idx])
            item["bm25_score"] = float(score)
            results.append(item)
        return results


print("📦 Building persistent ChromaDB vector index...")
embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
chroma_client = chromadb.PersistentClient(path=str(PATHS["vector_db"]))

# Recreate the collection so rerunning the notebook remains idempotent.
try:
    chroma_client.delete_collection("novastore_support_kb")
except Exception:
    pass

vector_collection = chroma_client.create_collection(
    name="novastore_support_kb",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"},
)

vector_collection.add(
    ids=[chunk["id"] for chunk in KB_CHUNKS],
    documents=[chunk["text"] for chunk in KB_CHUNKS],
    metadatas=[{
        "doc_id": chunk["doc_id"],
        "title": chunk["title"],
        "category": chunk["category"],
        "section": chunk["section"],
        "source": chunk["source"],
        "version": chunk["version"],
        "effective_date": chunk["effective_date"],
    } for chunk in KB_CHUNKS],
)

bm25_index = BM25Index(KB_CHUNKS)
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"✅ Vector index contains {vector_collection.count()} chunks")
print(f"✅ BM25 index contains {len(KB_CHUNKS)} chunks")

📦 Building persistent ChromaDB vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

  return datetime.utcnow().replace(tzinfo=utc)



...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  return datetime.utcnow().replace(tzinfo=utc)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  return datetime.utcnow().replace(tzinfo=utc)



✅ Vector index contains 50 chunks
✅ BM25 index contains 50 chunks


## Step 19 — Reciprocal Rank Fusion and Cross-Encoder Reranking

The advanced retrieval pipeline follows the course architecture:

```text
Query
  ├── Dense vector retrieval
  └── BM25 keyword retrieval
           ↓
Reciprocal Rank Fusion (RRF)
           ↓
Cross-Encoder precision reranker
           ↓
Best 3 grounded context chunks
```

RRF is rank-based, so the dense and sparse score scales do not need manual normalization.

In [23]:
# =====================================================================
# RAG STAGES 3–5 — HYBRID SEARCH, RRF, AND CROSS-ENCODER RERANKING
# =====================================================================

RERANKER = None


def vector_search(query: str, top_k: int = 10) -> list[dict]:
    result = vector_collection.query(
        query_texts=[query],
        n_results=min(top_k, vector_collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for chunk_id, document, metadata, distance in zip(
        result["ids"][0],
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({
            "id": chunk_id,
            "text": document,
            **metadata,
            "vector_distance": float(distance),
        })
    return hits


def reciprocal_rank_fusion(
    vector_hits: list[dict],
    bm25_hits: list[dict],
    k: int = 60,
    top_k: int = 10,
) -> list[dict]:
    rrf_scores: dict[str, float] = {}
    item_map: dict[str, dict] = {}

    for rank, item in enumerate(vector_hits, start=1):
        chunk_id = item["id"]
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
        item_map[chunk_id] = dict(item)

    for rank, item in enumerate(bm25_hits, start=1):
        chunk_id = item["id"]
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
        item_map[chunk_id] = {**item_map.get(chunk_id, {}), **item}

    ranked_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    fused = []
    for chunk_id in ranked_ids[:top_k]:
        item = item_map[chunk_id]
        item["rrf_score"] = float(rrf_scores[chunk_id])
        fused.append(item)
    return fused


def get_reranker() -> CrossEncoder:
    global RERANKER
    if RERANKER is None:
        print(f"🎯 Loading cross-encoder: {RERANKER_MODEL}")
        RERANKER = CrossEncoder(RERANKER_MODEL)
    return RERANKER


def cross_encoder_rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    if not candidates:
        return []

    try:
        reranker = get_reranker()
        pairs = [(query, candidate["text"]) for candidate in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, candidates), key=lambda item: float(item[0]), reverse=True)

        output = []
        for score, candidate in ranked[:top_k]:
            item = dict(candidate)
            item["reranker_score"] = float(score)
            output.append(item)
        return output

    except Exception as exc:
        logger.warning(f"Cross-encoder unavailable; preserving RRF ranking: {exc}")
        return candidates[:top_k]


def hybrid_retrieve(
    query: str,
    vector_top_k: int = 10,
    bm25_top_k: int = 10,
    fusion_top_k: int = 8,
    final_top_k: int = 3,
    verbose: bool = True,
) -> list[dict]:
    vector_hits = vector_search(query, top_k=vector_top_k)
    bm25_hits = bm25_index.search(query, top_k=bm25_top_k)
    fused = reciprocal_rank_fusion(vector_hits, bm25_hits, top_k=fusion_top_k)
    final_hits = cross_encoder_rerank(query, fused, top_k=final_top_k)

    if verbose:
        print(f"  Vector search : {len(vector_hits)} candidates")
        print(f"  BM25 search   : {len(bm25_hits)} candidates")
        print(f"  RRF fusion    : {len(fused)} candidates")
        print(f"  Final context : {len(final_hits)} chunks")

    return final_hits

## Step 20 — Prompt Construction and Llama-Grounded Answer Generation

The retriever first selects and reranks the most relevant approved company-policy chunks.  
The final answer is then generated using **Meta Llama 3.1 8B Instant through the Groq API**.

The LLM receives only:

- the customer's question;
- the top reranked policy chunks;
- strict instructions not to use outside knowledge;
- source labels that must be cited in the answer.

### Groq API key

In Google Colab, add a secret named:

```text
GROQ_API_KEY
```

from the **Secrets** panel on the left side of Colab.

For local execution, define the environment variable:

```bash
export GROQ_API_KEY="your-key"
```

If the key is missing or the API call fails, the code automatically uses the extractive grounded fallback instead of stopping the pipeline.


In [24]:
# =====================================================================
# RAG STAGE 6 — GROUNDED PROMPT AND LLAMA GENERATION THROUGH GROQ
# =====================================================================

GROQ_CLIENT = None


def get_groq_api_key() -> Optional[str]:
    """
    Read the Groq API key safely.

    Google Colab:
        Store the key in Colab Secrets using the name GROQ_API_KEY.

    Local execution:
        Export GROQ_API_KEY as an environment variable.

    The key is never printed or written to project artifacts.
    """
    api_key = os.getenv("GROQ_API_KEY")

    if api_key:
        return api_key.strip()

    if IN_COLAB:
        try:
            from google.colab import userdata
            secret = userdata.get("GROQ_API_KEY")
            if secret:
                return str(secret).strip()
        except Exception as exc:
            logger.warning(f"Could not read GROQ_API_KEY from Colab Secrets: {exc}")

    return None


def get_groq_client():
    """Create the Groq client lazily so the notebook can still run without a key."""
    global GROQ_CLIENT

    if GROQ_CLIENT is not None:
        return GROQ_CLIENT

    api_key = get_groq_api_key()
    if not api_key:
        return None

    try:
        from groq import Groq
        GROQ_CLIENT = Groq(api_key=api_key)
        return GROQ_CLIENT
    except Exception as exc:
        logger.warning(f"Groq client initialisation failed: {exc}")
        return None


def build_rag_prompt(query: str, context_docs: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source {index}] {doc['title']} — {doc['section']} "
        f"({doc['source']}):\n{doc['text']}"
        for index, doc in enumerate(context_docs, start=1)
    )

    return f"""You are NovaStore's customer-support assistant.

Use only the approved company-policy context below.
Do not use outside knowledge.
Do not invent dates, amounts, conditions, or procedures.
If the context is insufficient or conflicting, clearly state that a human agent is required.
Answer concisely and include source references such as [Source 1].

APPROVED CONTEXT:
{context}

CUSTOMER QUESTION:
{query}

GROUNDED ANSWER:"""


def extractive_grounded_answer(query: str, context_docs: list[dict]) -> str:
    """Safe fallback used when Groq is unavailable."""
    if not context_docs:
        return (
            "I cannot confirm the answer from the approved knowledge base. "
            "A human agent is required."
        )

    evidence_sentences = []
    for doc in context_docs[:2]:
        sentences = split_sentences(doc["text"])
        if sentences:
            evidence_sentences.append(sentences[0])
        if len(sentences) > 1:
            evidence_sentences.append(sentences[1])

    evidence = " ".join(evidence_sentences[:4])
    citations = " ".join(
        f"[Source {i}: {doc['source']} — {doc['section']}]"
        for i, doc in enumerate(context_docs, start=1)
    )
    return f"{evidence} {citations}".strip()


def ensure_source_reference(answer: str, context_docs: list[dict]) -> str:
    """
    Preserve traceability even if the LLM forgets to print a source label.
    """
    if not context_docs:
        return answer

    if "[Source" in answer:
        return answer

    source_suffix = " ".join(
        f"[Source {index}]"
        for index in range(1, min(len(context_docs), 3) + 1)
    )
    return f"{answer.rstrip()} {source_suffix}".strip()


def generate_grounded_answer(
    query: str,
    context_docs: list[dict],
    use_llm: bool = ENABLE_LLM_GENERATION,
) -> dict:
    """
    Generate the final answer from the reranked evidence.

    Preferred mode:
        Groq Chat Completions + llama-3.1-8b-instant.

    Fallback mode:
        Extractive answer built directly from approved evidence.
    """
    prompt = build_rag_prompt(query, context_docs)
    answer = None
    generation_mode = "extractive-grounded"

    if use_llm:
        client = get_groq_client()

        if client is None:
            logger.warning(
                "GROQ_API_KEY is unavailable. Using the extractive grounded fallback."
            )
        else:
            try:
                print(f"🤖 Generating with Groq model: {GENERATION_MODEL}")
                response = client.chat.completions.create(
                    model=GENERATION_MODEL,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "You are a careful customer-support assistant. "
                                "Use only the approved retrieved context. "
                                "Never guess. Cite the provided source numbers."
                            ),
                        },
                        {
                            "role": "user",
                            "content": prompt,
                        },
                    ],
                    temperature=GENERATION_TEMPERATURE,
                    max_tokens=GENERATION_MAX_TOKENS,
                )

                answer = response.choices[0].message.content.strip()
                answer = ensure_source_reference(answer, context_docs)
                generation_mode = f"groq:{GENERATION_MODEL}"

            except Exception as exc:
                logger.warning(
                    f"Groq Llama generation failed; using extractive fallback: {exc}"
                )

    if not answer:
        answer = extractive_grounded_answer(query, context_docs)

    return {
        "question": query,
        "answer": answer,
        "prompt": prompt,
        "generation_mode": generation_mode,
        "sources": [
            {
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "section": doc["section"],
                "source": doc["source"],
            }
            for doc in context_docs
        ],
    }


## Step 21 — Run a Complete RAG Query

This is the main customer-facing demonstration. It displays every retrieval stage, the reranked evidence, and the final sourced answer.

In [25]:
# =====================================================================
# END-TO-END RAG DEMONSTRATION
# =====================================================================

DEMO_QUERY = "I received a damaged product. What information do you need from me?"

print("=" * 70)
print(f"  CUSTOMER QUESTION: {DEMO_QUERY}")
print("=" * 70)

final_context = hybrid_retrieve(DEMO_QUERY, verbose=True)
rag_result = generate_grounded_answer(DEMO_QUERY, final_context)

print("\nTop reranked evidence:")
for index, doc in enumerate(final_context, start=1):
    print(f"\n[{index}] {doc['title']} — {doc['section']}")
    print(f"    Source: {doc['source']} | Doc ID: {doc['doc_id']}")
    print(f"    {doc['text'][:260]}...")

print("\n" + "-" * 70)
print("FINAL GROUNDED ANSWER")
print("-" * 70)
print(rag_result["answer"])

  CUSTOMER QUESTION: I received a damaged product. What information do you need from me?
🎯 Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

  return datetime.utcnow().replace(tzinfo=utc)



  Vector search : 10 candidates
  BM25 search   : 10 candidates
  RRF fusion    : 8 candidates
  Final context : 3 chunks


  return datetime.utcnow().replace(tzinfo=utc)



🤖 Generating with Groq model: llama-3.1-8b-instant

Top reranked evidence:

[1] Damaged, Incorrect, or Missing Items Policy — Reporting window
    Source: damaged_or_missing_items.md | Doc ID: KB-DMG-001
    Customers should report a damaged, incorrect, or missing item within **48 hours** of delivery....

[2] Refunds and Returns Policy — Return shipping
    Source: refunds_and_returns.md | Doc ID: KB-REF-001
    If the return is caused by a defective, damaged, or incorrect item, NovaStore covers the return shipping cost. For change-of-mind returns, the customer is responsible for return shipping....

[3] Damaged, Incorrect, or Missing Items Policy — Incorrect items
    Source: damaged_or_missing_items.md | Doc ID: KB-DMG-001
    If the wrong product was delivered, NovaStore provides return instructions and covers reasonable return shipping. A replacement is sent after the issue is confirmed, subject to stock availability....

------------------------------------------------------------

## Step 22 — Retrieval Evaluation

A small labelled test set measures whether the retrieval pipeline finds the expected policy document.

Metrics:

- **Hit@1:** expected document appears first.
- **Hit@3:** expected document appears anywhere in the final three.
- **MRR:** rewards higher rank of the first correct document.

This evaluates retrieval independently from generation, making debugging much easier.

In [26]:
# =====================================================================
# RETRIEVAL TEST SET
# =====================================================================

TEST_QUERIES = [
    {"query_id": "Q-001", "query": "How long do I have to return an unused item?", "expected_doc_id": "KB-REF-001"},
    {"query_id": "Q-002", "query": "When will my refund appear in my bank account?", "expected_doc_id": "KB-REF-001"},
    {"query_id": "Q-003", "query": "Can I change my delivery address after shipment?", "expected_doc_id": "KB-SHP-001"},
    {"query_id": "Q-004", "query": "My tracking has not moved for two days. What should I do?", "expected_doc_id": "KB-TRK-001"},
    {"query_id": "Q-005", "query": "Can I cancel an order that is still processing?", "expected_doc_id": "KB-CAN-001"},
    {"query_id": "Q-006", "query": "Why do I see two charges for one order?", "expected_doc_id": "KB-PAY-001"},
    {"query_id": "Q-007", "query": "I did not receive my password reset email.", "expected_doc_id": "KB-ACC-001"},
    {"query_id": "Q-008", "query": "The item arrived damaged. What evidence must I send?", "expected_doc_id": "KB-DMG-001"},
    {"query_id": "Q-009", "query": "Tracking says delivered but the package is missing.", "expected_doc_id": "KB-DMG-001"},
    {"query_id": "Q-010", "query": "I want to speak with a human agent.", "expected_doc_id": "KB-ESC-001"},
]

pd.DataFrame(TEST_QUERIES).to_csv(
    PATHS["knowledge_base"] / "test_queries.csv", index=False
)

In [27]:
# =====================================================================
# COMPUTE HIT@1, HIT@3, AND MRR
# =====================================================================

evaluation_rows = []

for test in TEST_QUERIES:
    hits = hybrid_retrieve(test["query"], final_top_k=3, verbose=False)
    retrieved_doc_ids = [hit["doc_id"] for hit in hits]

    try:
        correct_rank = retrieved_doc_ids.index(test["expected_doc_id"]) + 1
    except ValueError:
        correct_rank = None

    evaluation_rows.append({
        "query_id": test["query_id"],
        "query": test["query"],
        "expected_doc_id": test["expected_doc_id"],
        "retrieved_doc_ids": " | ".join(retrieved_doc_ids),
        "correct_rank": correct_rank,
        "hit_at_1": int(correct_rank == 1),
        "hit_at_3": int(correct_rank is not None and correct_rank <= 3),
        "reciprocal_rank": 1.0 / correct_rank if correct_rank else 0.0,
    })

evaluation_df = pd.DataFrame(evaluation_rows)
retrieval_metrics = {
    "hit_at_1": float(evaluation_df["hit_at_1"].mean()),
    "hit_at_3": float(evaluation_df["hit_at_3"].mean()),
    "mrr": float(evaluation_df["reciprocal_rank"].mean()),
    "queries": len(evaluation_df),
}

evaluation_path = PATHS["quality"] / "retrieval_evaluation.csv"
evaluation_df.to_csv(evaluation_path, index=False)

print("=" * 70)
print("  RETRIEVAL EVALUATION")
print("=" * 70)
print(f"Hit@1 : {retrieval_metrics['hit_at_1']:.3f}")
print(f"Hit@3 : {retrieval_metrics['hit_at_3']:.3f}")
print(f"MRR   : {retrieval_metrics['mrr']:.3f}")
print(f"Saved : {evaluation_path}")
print("=" * 70)
display(evaluation_df)

  return datetime.utcnow().replace(tzinfo=utc)



  RETRIEVAL EVALUATION
Hit@1 : 0.900
Hit@3 : 1.000
MRR   : 0.950
Saved : /content/supportai_final_project/quality_reports/retrieval_evaluation.csv


,query_id,query,expected_doc_id,retrieved_doc_ids,correct_rank,hit_at_1,hit_at_3,reciprocal_rank
0,Q-001,How long do I have to return an unused item?,KB-REF-001,KB-REF-001 | KB-REF-001 | KB-DMG-001,1,1,1,1.0
1,Q-002,When will my refund appear in my bank account?,KB-REF-001,KB-REF-001 | KB-REF-001 | KB-PAY-001,1,1,1,1.0
2,Q-003,Can I change my delivery address after shipment?,KB-SHP-001,KB-SHP-001 | KB-CAN-001 | KB-CAN-001,1,1,1,1.0
3,Q-004,My tracking has not moved for two days. What s...,KB-TRK-001,KB-SHP-001 | KB-TRK-001 | KB-TRK-001,2,0,1,0.5
4,Q-005,Can I cancel an order that is still processing?,KB-CAN-001,KB-CAN-001 | KB-CAN-001 | KB-CAN-001,1,1,1,1.0
5,Q-006,Why do I see two charges for one order?,KB-PAY-001,KB-PAY-001 | KB-PAY-001 | KB-TRK-001,1,1,1,1.0
6,Q-007,I did not receive my password reset email.,KB-ACC-001,KB-ACC-001 | KB-ACC-001 | KB-ACC-001,1,1,1,1.0
7,Q-008,The item arrived damaged. What evidence must I...,KB-DMG-001,KB-DMG-001 | KB-REF-001 | KB-REF-001,1,1,1,1.0
8,Q-009,Tracking says delivered but the package is mis...,KB-DMG-001,KB-DMG-001 | KB-SHP-001 | KB-SHP-001,1,1,1,1.0
9,Q-010,I want to speak with a human agent.,KB-ESC-001,KB-ESC-001 | KB-ESC-001 | KB-CAN-001,1,1,1,1.0


## Step 23 — Official OpenLineage Audit Trail

This stage uses the official `openlineage-python` client and emits valid `RunEvent` objects with START and COMPLETE states to a JSONL file transport.

In [28]:
# =====================================================================
# OFFICIAL OPENLINEAGE PYTHON CLIENT — FILE TRANSPORT
# =====================================================================

from openlineage.client import OpenLineageClient
from openlineage.client.event_v2 import Dataset, Job, Run, RunEvent, RunState
from openlineage.client.transport.file import FileConfig, FileTransport
from openlineage.client.uuid import generate_new_uuid

openlineage_path = PATHS["lineage"] / "openlineage_events.jsonl"
openlineage_client = OpenLineageClient(
    transport=FileTransport(
        FileConfig(log_file_path=str(openlineage_path), append=True)
    )
)


def emit_openlineage_run(job_name, state, run_id, inputs, outputs):
    event = RunEvent(
        eventType=state,
        eventTime=datetime.now(timezone.utc).isoformat(),
        run=Run(runId=run_id),
        job=Job(namespace="supportai", name=job_name),
        producer="https://example.org/supportai-final-project",
        inputs=[Dataset(namespace="supportai", name=name) for name in inputs],
        outputs=[Dataset(namespace="supportai", name=name) for name in outputs],
    )
    openlineage_client.emit(event)
    print(f"[OPENLINEAGE] {state.value:<8} | {job_name}")


lineage_jobs = [
    ("kafka_ingestion", [DATASET_ID], ["bronze.customer_support_tickets"]),
    ("quality_gate", ["bronze.customer_support_tickets"], ["quality.gx_validation"]),
    ("lakehouse_transform", ["bronze.customer_support_tickets"], ["silver.customer_support_tickets", "gold.support_metrics"]),
    ("rag_indexing", ["knowledge_base.novastore_policies"], ["vector_db.novastore_support_kb"]),
    ("rag_evaluation", ["vector_db.novastore_support_kb"], ["quality.retrieval_evaluation"]),
    ("rag_serving", ["vector_db.novastore_support_kb"], ["gold.rag_answer"]),
]

for job_name, inputs, outputs in lineage_jobs:
    run_id = str(generate_new_uuid())
    emit_openlineage_run(job_name, RunState.START, run_id, inputs, outputs)
    emit_openlineage_run(job_name, RunState.COMPLETE, run_id, inputs, outputs)

print(f"✅ Official OpenLineage RunEvents saved to: {openlineage_path}")
print(openlineage_path.read_text(encoding="utf-8").splitlines()[0][:500])


  return datetime.utcnow().replace(tzinfo=utc)



[OPENLINEAGE] START    | kafka_ingestion
[OPENLINEAGE] COMPLETE | kafka_ingestion
[OPENLINEAGE] START    | quality_gate
[OPENLINEAGE] COMPLETE | quality_gate
[OPENLINEAGE] START    | lakehouse_transform
[OPENLINEAGE] COMPLETE | lakehouse_transform
[OPENLINEAGE] START    | rag_indexing
[OPENLINEAGE] COMPLETE | rag_indexing
[OPENLINEAGE] START    | rag_evaluation
[OPENLINEAGE] COMPLETE | rag_evaluation
[OPENLINEAGE] START    | rag_serving
[OPENLINEAGE] COMPLETE | rag_serving
✅ Official OpenLineage RunEvents saved to: /content/supportai_final_project/lineage_events/openlineage_events.jsonl
{"eventTime": "2026-07-02T21:00:16.835536+00:00", "eventType": "START", "inputs": [{"facets": {}, "name": "bitext/Bitext-customer-support-llm-chatbot-training-dataset", "namespace": "supportai"}], "job": {"facets": {}, "name": "kafka_ingestion", "namespace": "supportai"}, "outputs": [{"facets": {}, "name": "bronze.customer_support_tickets", "namespace": "supportai"}], "producer": "https://example.org/su

## Step 24 — Save the Final RAG Answer to Gold

The generated answer and its sources are stored as a Gold serving table. This demonstrates how an AI output becomes a governed enterprise data asset rather than disappearing after it is displayed.

In [29]:
# =====================================================================
# GOLD AI SERVING OUTPUT — RAG ANSWER WITH SOURCE METADATA
# =====================================================================

answer_record = {
    "answer_id": str(uuid.uuid4()),
    "ticket_id": "DEMO-TICKET-001",
    "question": rag_result["question"],
    "answer": rag_result["answer"],
    "source_doc_ids": json.dumps([s["doc_id"] for s in rag_result["sources"]]),
    "source_files": json.dumps([s["source"] for s in rag_result["sources"]]),
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "generation_mode": rag_result.get("generation_mode", "extractive-grounded"),
}

answers_pd = pd.DataFrame([answer_record])
answers_spark = spark.createDataFrame(answers_pd)
gold_answers_path = str(PATHS["gold"] / "rag_answers_delta")
answers_spark.write.format("delta").mode("append").save(gold_answers_path)

print("✅ RAG answer saved to Gold Delta table")
print(f"Path: {gold_answers_path}")
spark.read.format("delta").load(gold_answers_path).show(truncate=60)

✅ RAG answer saved to Gold Delta table
Path: /content/supportai_final_project/data/gold/rag_answers_delta
+------------------------------------+---------------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------+------------------------------------------------------------+--------------------------------+-------------------------+
|                           answer_id|      ticket_id|                                                    question|                                                      answer|                            source_doc_ids|                                                source_files|                    generated_at|          generation_mode|
+------------------------------------+---------------+------------------------------------------------------------+------------------------------------------------------------+--------------------------------------

## Step 25 — Generate an Executable Airflow DAG

The DAG uses `BashOperator` tasks that invoke real CLI stages. A failed quality command returns a non-zero exit code, so Airflow blocks every downstream task.

In [30]:
# =====================================================================
# EXECUTABLE AIRFLOW DAG — REAL COMMANDS, NOT PLACEHOLDERS
# =====================================================================

airflow_dag_code = 'from __future__ import annotations\n\nimport os\nfrom datetime import datetime, timedelta\n\nfrom airflow import DAG\ntry:\n    # Airflow 3.x\n    from airflow.providers.standard.operators.bash import BashOperator\nexcept ImportError:\n    # Airflow 2.x compatibility\n    from airflow.operators.bash import BashOperator\n\nPROJECT_ROOT = os.getenv("SUPPORTAI_PROJECT_ROOT", "/opt/airflow/supportai")\nPYTHON_BIN = os.getenv("SUPPORTAI_PYTHON_BIN", "python")\nPIPELINE = f"{PROJECT_ROOT}/supportai_pipeline.py"\nCOMMON_ENV = {\n    "SUPPORTAI_PROJECT_ROOT": PROJECT_ROOT,\n    "SUPPORTAI_BOOTSTRAP_SERVERS": os.getenv("SUPPORTAI_BOOTSTRAP_SERVERS", "redpanda:9092"),\n    "PYTHONPATH": PROJECT_ROOT,\n}\n\nDEFAULT_ARGS = {\n    "owner": "supportai-team",\n    "retries": 1,\n    "retry_delay": timedelta(minutes=1),\n}\n\nwith DAG(\n    dag_id="supportai_final_pipeline",\n    description="Real Kafka -> blocking quality gate -> Delta Lakehouse -> Hybrid RAG",\n    start_date=datetime(2026, 1, 1),\n    schedule=None,\n    catchup=False,\n    default_args=DEFAULT_ARGS,\n    tags=["final-project", "kafka", "delta", "rag", "openlineage"],\n) as dag:\n\n    prepare_source_data = BashOperator(\n        task_id="prepare_source_data",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} prepare-data",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    publish_kafka_events = BashOperator(\n        task_id="publish_kafka_events",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} kafka-producer",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    consume_validate_write_bronze = BashOperator(\n        task_id="consume_validate_write_bronze",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} kafka-consumer-to-bronze",\n        env=COMMON_ENV,\n        append_env=True,\n        execution_timeout=timedelta(minutes=10),\n    )\n\n    quality_gate = BashOperator(\n        task_id="great_expectations_quality_gate",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} quality-gate",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    build_lakehouse = BashOperator(\n        task_id="build_delta_lakehouse",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} build-lakehouse",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    build_rag_index = BashOperator(\n        task_id="build_hybrid_rag_index",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} build-rag",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    evaluate_rag = BashOperator(\n        task_id="evaluate_rag",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} evaluate-rag",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    publish_serving_output = BashOperator(\n        task_id="publish_serving_output",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} publish-answer",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    verify_deliverables = BashOperator(\n        task_id="verify_deliverables",\n        bash_command=f"{PYTHON_BIN} {PIPELINE} verify",\n        env=COMMON_ENV,\n        append_env=True,\n    )\n\n    (\n        prepare_source_data\n        >> publish_kafka_events\n        >> consume_validate_write_bronze\n        >> quality_gate\n        >> build_lakehouse\n        >> build_rag_index\n        >> evaluate_rag\n        >> publish_serving_output\n        >> verify_deliverables\n    )\n'
dag_path = PATHS["dags"] / "supportai_final_pipeline.py"
dag_path.write_text(airflow_dag_code, encoding="utf-8")
print(f"✅ Executable Airflow DAG generated: {dag_path}")
print("Every task invokes a real supportai_pipeline.py command.")
print("\n".join(airflow_dag_code.splitlines()[:45]))


✅ Executable Airflow DAG generated: /content/supportai_final_project/dags/supportai_final_pipeline.py
Every task invokes a real supportai_pipeline.py command.
from __future__ import annotations

import os
from datetime import datetime, timedelta

from airflow import DAG
try:
    # Airflow 3.x
    from airflow.providers.standard.operators.bash import BashOperator
except ImportError:
    # Airflow 2.x compatibility
    from airflow.operators.bash import BashOperator

PROJECT_ROOT = os.getenv("SUPPORTAI_PROJECT_ROOT", "/opt/airflow/supportai")
PYTHON_BIN = os.getenv("SUPPORTAI_PYTHON_BIN", "python")
PIPELINE = f"{PROJECT_ROOT}/supportai_pipeline.py"
COMMON_ENV = {
    "SUPPORTAI_PROJECT_ROOT": PROJECT_ROOT,
    "SUPPORTAI_BOOTSTRAP_SERVERS": os.getenv("SUPPORTAI_BOOTSTRAP_SERVERS", "redpanda:9092"),
    "PYTHONPATH": PROJECT_ROOT,
}

DEFAULT_ARGS = {
    "owner": "supportai-team",
    "retries": 1,
    "retry_delay": timedelta(minutes=1),
}

with DAG(
    dag_id="supportai_final_pipeline"

## Step 26 — Generate Real Kafka/Redpanda Artifacts

These files run outside Colab with Docker. The consumer performs Pydantic validation, records partition/offset metadata, writes the Bronze Delta table, and routes rejected records to Quarantine.

In [31]:
# =====================================================================
# REAL REDPANDA / KAFKA PRODUCER + VALIDATING CONSUMER
# =====================================================================

real_kafka_dir = PROJECT_ROOT / "optional_real_kafka"
real_kafka_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    "docker-compose.yml": 'services:\n  redpanda:\n    image: redpandadata/redpanda:latest\n    command:\n      - redpanda\n      - start\n      - --overprovisioned\n      - --smp=1\n      - --memory=1G\n      - --reserve-memory=0M\n      - --node-id=0\n      - --check=false\n      - --kafka-addr=internal://0.0.0.0:29092,external://0.0.0.0:9092\n      - --advertise-kafka-addr=internal://redpanda:29092,external://localhost:9092\n    ports:\n      - "9092:9092"\n      - "9644:9644"\n    healthcheck:\n      test: ["CMD-SHELL", "rpk cluster health --exit-when-healthy"]\n      interval: 5s\n      timeout: 5s\n      retries: 20\n\n  console:\n    image: redpandadata/console:latest\n    depends_on:\n      redpanda:\n        condition: service_healthy\n    environment:\n      KAFKA_BROKERS: redpanda:29092\n    ports:\n      - "8080:8080"\n',
    "producer.py": '#!/usr/bin/env python3\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom supportai_pipeline import kafka_producer\n\nif __name__ == "__main__":\n    kafka_producer()\n',
    "consumer.py": '#!/usr/bin/env python3\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom supportai_pipeline import kafka_consumer_to_bronze\n\nif __name__ == "__main__":\n    kafka_consumer_to_bronze()\n',
}
for filename, content in artifacts.items():
    (real_kafka_dir / filename).write_text(content, encoding="utf-8")

pipeline_target = PROJECT_ROOT / "supportai_pipeline.py"
pipeline_target.write_text('#!/usr/bin/env python3\n"""SupportAI executable pipeline.\n\nThis command-line module is the implementation used by the Airflow DAG. It\ncontains real stage functions rather than placeholder tasks:\n\nprepare-data -> kafka-producer -> kafka-consumer-to-bronze -> quality-gate\n-> build-lakehouse -> build-rag -> evaluate-rag -> publish-answer\n\nEnvironment variables:\n  SUPPORTAI_PROJECT_ROOT      Root folder of the project.\n  SUPPORTAI_BOOTSTRAP_SERVERS Kafka/Redpanda address (default localhost:9092).\n  SUPPORTAI_SAMPLE_SIZE       Number of Bitext records (default 200).\n  SUPPORTAI_FORCE_QUALITY_FAILURE=true to demonstrate a blocking quality gate.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport random\nimport re\nimport sys\nimport time\nimport uuid\nfrom contextlib import contextmanager\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\nfrom pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator\n\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\n\nDATASET_ID = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"\nTOPIC = "customer_support_tickets_raw"\nEMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"\nRERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"\nALLOWED_CHANNELS = {"chat", "email", "mobile_app", "web_form"}\nALLOWED_PRIORITIES = {"low", "medium", "high"}\nALLOWED_STATUSES = {"open", "pending", "closed", "escalated"}\n\nPROJECT_ROOT = Path(os.getenv("SUPPORTAI_PROJECT_ROOT", Path(__file__).resolve().parent))\nBOOTSTRAP_SERVERS = os.getenv("SUPPORTAI_BOOTSTRAP_SERVERS", "localhost:9092")\nSAMPLE_SIZE = int(os.getenv("SUPPORTAI_SAMPLE_SIZE", "200"))\n\nPATHS = {\n    "raw": PROJECT_ROOT / "data" / "raw",\n    "bronze": PROJECT_ROOT / "data" / "bronze",\n    "silver": PROJECT_ROOT / "data" / "silver",\n    "gold": PROJECT_ROOT / "data" / "gold",\n    "quarantine": PROJECT_ROOT / "data" / "quarantine",\n    "kb": PROJECT_ROOT / "knowledge_base",\n    "vector": PROJECT_ROOT / "vector_db",\n    "quality": PROJECT_ROOT / "quality_reports",\n    "lineage": PROJECT_ROOT / "lineage_events",\n}\nfor path in PATHS.values():\n    path.mkdir(parents=True, exist_ok=True)\n\n\nclass SupportTicketContract(BaseModel):\n    """Strict-enough event contract for Kafka ingestion."""\n\n    model_config = ConfigDict(strict=False, extra="forbid")\n\n    ticket_id: str\n    customer_id: str\n    question: str = Field(min_length=5, max_length=2000)\n    category: str\n    intent: str\n    channel: str\n    priority: str\n    status: str\n    event_timestamp: datetime\n    reference_response: str = ""\n    flags: str = ""\n\n    @field_validator("ticket_id", "customer_id", "category", "intent")\n    @classmethod\n    def required_string(cls, value: str) -> str:\n        value = str(value).strip()\n        if not value or value.lower() in {"nan", "none"}:\n            raise ValueError("required field cannot be empty")\n        return value\n\n    @field_validator("question")\n    @classmethod\n    def normalise_question(cls, value: str) -> str:\n        value = re.sub(r"\\s+", " ", str(value)).strip()\n        if len(value) < 5:\n            raise ValueError("question must contain at least 5 characters")\n        return value\n\n    @field_validator("channel")\n    @classmethod\n    def valid_channel(cls, value: str) -> str:\n        value = str(value).strip().lower()\n        if value not in ALLOWED_CHANNELS:\n            raise ValueError(f"channel must be one of {sorted(ALLOWED_CHANNELS)}")\n        return value\n\n    @field_validator("priority")\n    @classmethod\n    def valid_priority(cls, value: str) -> str:\n        value = str(value).strip().lower()\n        if value not in ALLOWED_PRIORITIES:\n            raise ValueError(f"priority must be one of {sorted(ALLOWED_PRIORITIES)}")\n        return value\n\n    @field_validator("status")\n    @classmethod\n    def valid_status(cls, value: str) -> str:\n        value = str(value).strip().lower()\n        if value not in ALLOWED_STATUSES:\n            raise ValueError(f"status must be one of {sorted(ALLOWED_STATUSES)}")\n        return value\n\n\ndef _now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef _json_safe(record: dict[str, Any]) -> dict[str, Any]:\n    result: dict[str, Any] = {}\n    for key, value in record.items():\n        if pd.isna(value):\n            result[key] = None\n        elif isinstance(value, (np.integer,)):\n            result[key] = int(value)\n        elif isinstance(value, (np.floating,)):\n            result[key] = float(value)\n        else:\n            result[key] = value\n    return result\n\n\ndef _write_json(path: Path, payload: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n\n\ndef _write_jsonl(path: Path, records: Iterable[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for record in records:\n            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\\n")\n\n\ndef _read_jsonl(path: Path) -> list[dict[str, Any]]:\n    records: list[dict[str, Any]] = []\n    with path.open("r", encoding="utf-8") as handle:\n        for line in handle:\n            if line.strip():\n                records.append(json.loads(line))\n    return records\n\n\ndef _lineage_client():\n    """Return the official OpenLineage client with file transport."""\n    from openlineage.client import OpenLineageClient\n    from openlineage.client.transport.file import FileConfig, FileTransport\n\n    output = PATHS["lineage"] / "openlineage_events.jsonl"\n    return OpenLineageClient(\n        transport=FileTransport(FileConfig(log_file_path=str(output), append=True))\n    )\n\n\n@contextmanager\ndef lineage_run(job_name: str, inputs: list[str], outputs: list[str]):\n    """Emit official OpenLineage START/COMPLETE/FAIL RunEvents."""\n    from openlineage.client.event_v2 import Dataset, Job, Run, RunEvent, RunState\n    from openlineage.client.uuid import generate_new_uuid\n\n    client = _lineage_client()\n    run = Run(runId=str(generate_new_uuid()))\n    job = Job(namespace="supportai", name=job_name)\n    input_datasets = [Dataset(namespace="supportai", name=name) for name in inputs]\n    output_datasets = [Dataset(namespace="supportai", name=name) for name in outputs]\n    producer = "https://example.org/supportai-final-project"\n\n    def emit(state):\n        client.emit(\n            RunEvent(\n                eventType=state,\n                eventTime=_now(),\n                run=run,\n                job=job,\n                producer=producer,\n                inputs=input_datasets,\n                outputs=output_datasets,\n            )\n        )\n\n    emit(RunState.START)\n    try:\n        yield\n    except Exception:\n        emit(RunState.FAIL)\n        raise\n    else:\n        emit(RunState.COMPLETE)\n\n\ndef spark_session(app_name: str):\n    from delta import configure_spark_with_delta_pip\n    from pyspark.sql import SparkSession\n\n    builder = (\n        SparkSession.builder\n        .appName(app_name)\n        .master(os.getenv("SUPPORTAI_SPARK_MASTER", "local[*]"))\n        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\n        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\n        .config("spark.sql.shuffle.partitions", "4")\n    )\n    spark = configure_spark_with_delta_pip(builder).getOrCreate()\n    spark.sparkContext.setLogLevel("ERROR")\n    return spark\n\n\ndef prepare_data() -> None:\n    with lineage_run("prepare_source_data", [DATASET_ID], ["raw.bitext_support_tickets"]):\n        from datasets import load_dataset\n\n        print(f"Loading {DATASET_ID} ...")\n        dataset = load_dataset(DATASET_ID, split="train")\n        df = dataset.to_pandas()\n        sample_n = min(SAMPLE_SIZE, len(df))\n        df = df.sample(sample_n, random_state=SEED).reset_index(drop=True)\n\n        base_time = datetime.now(timezone.utc) - timedelta(minutes=sample_n)\n        channels = ["chat", "email", "mobile_app", "web_form"]\n        priorities = ["low", "medium", "high"]\n\n        enriched = pd.DataFrame({\n            "ticket_id": [f"T-{i+1:05d}" for i in range(sample_n)],\n            "customer_id": [f"C-{1000 + (i % 80):04d}" for i in range(sample_n)],\n            "question": df["instruction"].astype(str),\n            "category": df["category"].astype(str),\n            "intent": df["intent"].astype(str),\n            "channel": [channels[i % len(channels)] for i in range(sample_n)],\n            "priority": [priorities[i % len(priorities)] for i in range(sample_n)],\n            "status": "open",\n            "event_timestamp": [\n                (base_time + timedelta(minutes=i)).isoformat() for i in range(sample_n)\n            ],\n            "reference_response": df["response"].astype(str),\n            "flags": df["flags"].fillna("").astype(str),\n        })\n\n        # Deliberate bad records prove validation and quarantine behaviour.\n        if sample_n >= 6:\n            enriched.loc[0, "ticket_id"] = ""\n            enriched.loc[1, "question"] = ""\n            enriched.loc[2, "channel"] = "fax"\n            enriched.loc[3, "priority"] = "urgent"\n            enriched.loc[4, "event_timestamp"] = "not-a-date"\n            enriched.loc[5, "ticket_id"] = enriched.loc[6, "ticket_id"] if sample_n > 6 else "T-DUP"\n\n        output = PATHS["raw"] / "bitext_support_tickets_enriched.csv"\n        enriched.to_csv(output, index=False)\n        print(f"Prepared {len(enriched)} events: {output}")\n\n\ndef kafka_producer() -> None:\n    with lineage_run("kafka_producer", ["raw.bitext_support_tickets"], [f"kafka.{TOPIC}"]):\n        from confluent_kafka import Producer\n\n        source = PATHS["raw"] / "bitext_support_tickets_enriched.csv"\n        if not source.exists():\n            raise FileNotFoundError(f"Run prepare-data first: {source}")\n        df = pd.read_csv(source, keep_default_na=False)\n        producer = Producer({"bootstrap.servers": BOOTSTRAP_SERVERS, "acks": "all"})\n\n        def delivery_report(error, message):\n            if error is not None:\n                raise RuntimeError(f"Kafka delivery failed: {error}")\n\n        for index, record in enumerate(df.to_dict(orient="records"), start=1):\n            clean = _json_safe(record)\n            key = str(clean.get("ticket_id") or f"missing-{index}")\n            producer.produce(\n                TOPIC,\n                key=key,\n                value=json.dumps(clean, ensure_ascii=False),\n                on_delivery=delivery_report,\n            )\n            producer.poll(0)\n            if index <= 5 or index % 50 == 0:\n                print(f"Produced {index}/{len(df)} ticket={key}")\n\n        # A control event lets the finite demonstration consumer terminate cleanly.\n        producer.produce(\n            TOPIC,\n            key="__control__",\n            value=json.dumps({"_event_type": "END_OF_STREAM", "expected_records": len(df)}),\n            on_delivery=delivery_report,\n        )\n        producer.flush()\n        print(f"Published {len(df)} events plus END_OF_STREAM to {TOPIC}")\n\n\ndef _write_bronze_delta(records: list[dict[str, Any]]) -> Path:\n    from pyspark.sql.types import IntegerType, StringType, StructField, StructType\n\n    schema = StructType([\n        StructField("ticket_id", StringType(), True),\n        StructField("customer_id", StringType(), True),\n        StructField("question", StringType(), True),\n        StructField("category", StringType(), True),\n        StructField("intent", StringType(), True),\n        StructField("channel", StringType(), True),\n        StructField("priority", StringType(), True),\n        StructField("status", StringType(), True),\n        StructField("event_timestamp", StringType(), True),\n        StructField("reference_response", StringType(), True),\n        StructField("flags", StringType(), True),\n        StructField("kafka_partition", IntegerType(), True),\n        StructField("kafka_offset", IntegerType(), True),\n        StructField("ingested_at", StringType(), True),\n    ])\n    columns = [field.name for field in schema.fields]\n    normalised: list[dict[str, Any]] = []\n    for item in records:\n        row = {name: item.get(name) for name in columns}\n        for name in columns:\n            if name not in {"kafka_partition", "kafka_offset"}:\n                row[name] = "" if row[name] is None else str(row[name])\n        row["kafka_partition"] = int(row.get("kafka_partition", -1))\n        row["kafka_offset"] = int(row.get("kafka_offset", -1))\n        normalised.append(row)\n\n    spark = spark_session("SupportAI_Kafka_Consumer_Bronze")\n    path = PATHS["bronze"] / "tickets_delta"\n    spark.createDataFrame(normalised, schema=schema).write.format("delta").mode("overwrite").save(str(path))\n    print(f"Bronze Delta rows: {spark.read.format(\'delta\').load(str(path)).count()}")\n    spark.stop()\n    return path\n\n\ndef kafka_consumer_to_bronze() -> None:\n    with lineage_run(\n        "kafka_consumer_to_bronze",\n        [f"kafka.{TOPIC}"],\n        ["bronze.customer_support_tickets", "quarantine.contract_violations"],\n    ):\n        from confluent_kafka import Consumer, KafkaError\n\n        group_id = os.getenv("SUPPORTAI_KAFKA_GROUP_ID", f"supportai-{uuid.uuid4()}")\n        consumer = Consumer({\n            "bootstrap.servers": BOOTSTRAP_SERVERS,\n            "group.id": group_id,\n            "auto.offset.reset": "earliest",\n            "enable.auto.commit": False,\n        })\n        consumer.subscribe([TOPIC])\n\n        bronze: list[dict[str, Any]] = []\n        valid: list[dict[str, Any]] = []\n        rejected: list[dict[str, Any]] = []\n        seen_ticket_ids: set[str] = set()\n        idle_polls = 0\n        max_idle = int(os.getenv("SUPPORTAI_MAX_IDLE_POLLS", "120"))\n\n        try:\n            while True:\n                message = consumer.poll(1.0)\n                if message is None:\n                    idle_polls += 1\n                    if idle_polls >= max_idle:\n                        raise TimeoutError("END_OF_STREAM was not received from Kafka")\n                    continue\n                idle_polls = 0\n                if message.error():\n                    if message.error().code() == KafkaError._PARTITION_EOF:\n                        continue\n                    raise RuntimeError(str(message.error()))\n\n                event = json.loads(message.value().decode("utf-8"))\n                if event.get("_event_type") == "END_OF_STREAM":\n                    print(f"Received END_OF_STREAM after {len(bronze)} data events")\n                    break\n\n                raw = dict(event)\n                raw.update({\n                    "kafka_partition": message.partition(),\n                    "kafka_offset": message.offset(),\n                    "ingested_at": _now(),\n                })\n                bronze.append(raw)\n\n                try:\n                    validated = SupportTicketContract(**event).model_dump(mode="json")\n                    ticket_id = validated["ticket_id"]\n                    if ticket_id in seen_ticket_ids:\n                        failed = raw.copy()\n                        failed["rejection_reason"] = f"duplicate ticket_id: {ticket_id}"\n                        rejected.append(failed)\n                        continue\n                    seen_ticket_ids.add(ticket_id)\n                    validated.update({\n                        "kafka_partition": message.partition(),\n                        "kafka_offset": message.offset(),\n                        "ingested_at": raw["ingested_at"],\n                    })\n                    valid.append(validated)\n                except ValidationError as exc:\n                    failed = raw.copy()\n                    failed["rejection_reason"] = exc.json()\n                    rejected.append(failed)\n        finally:\n            consumer.close()\n\n        _write_jsonl(PATHS["bronze"] / "bronze_events.jsonl", bronze)\n        _write_jsonl(PATHS["bronze"] / "contract_valid_events.jsonl", valid)\n        _write_jsonl(PATHS["quarantine"] / "contract_violations.jsonl", rejected)\n        _write_bronze_delta(bronze)\n        _write_json(PATHS["bronze"] / "ingestion_summary.json", {\n            "bronze_events": len(bronze),\n            "contract_valid": len(valid),\n            "quarantined": len(rejected),\n            "topic": TOPIC,\n            "consumer_group": group_id,\n        })\n        print(f"Valid={len(valid)} Quarantined={len(rejected)} Bronze={len(bronze)}")\n\n\ndef quality_gate() -> None:\n    with lineage_run(\n        "great_expectations_quality_gate",\n        ["bronze.contract_valid_events"],\n        ["quality.great_expectations_result"],\n    ):\n        import great_expectations as gx\n\n        source = PATHS["bronze"] / "contract_valid_events.jsonl"\n        records = _read_jsonl(source)\n        df = pd.DataFrame(records)\n        if df.empty:\n            raise RuntimeError("Quality gate received no valid records")\n\n        if os.getenv("SUPPORTAI_FORCE_QUALITY_FAILURE", "false").lower() == "true":\n            df.loc[df.index[0], "channel"] = "invalid-channel-for-gate-demo"\n\n        validator = gx.from_pandas(df.copy())\n        expectations = [\n            validator.expect_column_values_to_not_be_null("ticket_id"),\n            validator.expect_column_values_to_be_unique("ticket_id"),\n            validator.expect_column_values_to_not_be_null("question"),\n            validator.expect_column_values_to_be_in_set("channel", sorted(ALLOWED_CHANNELS)),\n            validator.expect_column_values_to_be_in_set("priority", sorted(ALLOWED_PRIORITIES)),\n            validator.expect_column_value_lengths_to_be_between("question", min_value=5, max_value=2000),\n        ]\n        validation = validator.validate(result_format="SUMMARY")\n        try:\n            report = validation.to_json_dict()\n        except AttributeError:\n            report = dict(validation)\n        def _expectation_passed(item):\n            if hasattr(item, "success"):\n                return bool(item.success)\n            if isinstance(item, dict):\n                return bool(item.get("success", False))\n            return False\n\n        passed = bool(report.get("success", all(_expectation_passed(item) for item in expectations)))\n        report["quality_gate_passed"] = passed\n        report["rows_evaluated"] = len(df)\n        output = PATHS["quality"] / "great_expectations_validation.json"\n        _write_json(output, report)\n        print(f"Great Expectations gate: {\'PASSED\' if passed else \'FAILED\'}")\n\n        # A non-zero process exit makes BashOperator fail and blocks every downstream task.\n        if not passed:\n            _write_jsonl(PATHS["quarantine"] / "quality_gate_failed_batch.jsonl", records)\n            raise RuntimeError(f"Quality gate failed; downstream pipeline blocked. Report: {output}")\n\n\ndef build_lakehouse() -> None:\n    with lineage_run(\n        "build_delta_lakehouse",\n        ["bronze.contract_valid_events"],\n        ["silver.customer_support_tickets", "gold.support_metrics"],\n    ):\n        from delta.tables import DeltaTable\n        from pyspark.sql import functions as F\n        from pyspark.sql.types import StringType, StructField, StructType\n\n        records = _read_jsonl(PATHS["bronze"] / "contract_valid_events.jsonl")\n        df = pd.DataFrame(records)\n        df["question"] = (\n            df["question"].astype(str)\n            .str.replace(r"\\s+", " ", regex=True)\n            .str.strip()\n            .str.replace(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}", "[EMAIL_REDACTED]", regex=True)\n        )\n        df = df.drop_duplicates(subset=["ticket_id"], keep="last").reset_index(drop=True)\n\n        spark = spark_session("SupportAI_Lakehouse")\n        silver_path = PATHS["silver"] / "tickets_delta"\n        silver_df = spark.createDataFrame(df.astype(str))\n        silver_df.write.format("delta").mode("overwrite").save(str(silver_path))\n\n        # Explicit schema-enforcement proof: appending a new column without mergeSchema must fail.\n        enforcement: dict[str, Any] = {"passed": False, "attempted_change": "unexpected_new_column"}\n        try:\n            bad_df = silver_df.limit(1).withColumn("unexpected_new_column", F.lit("must_be_rejected"))\n            bad_df.write.format("delta").mode("append").save(str(silver_path))\n        except Exception as exc:\n            enforcement.update({"passed": True, "blocked": True, "error": str(exc)[:4000]})\n            print("Schema enforcement correctly blocked an incompatible append")\n        else:\n            enforcement.update({"blocked": False, "error": "Delta unexpectedly accepted schema drift"})\n        _write_json(PATHS["quality"] / "delta_schema_enforcement_evidence.json", enforcement)\n        if not enforcement["passed"]:\n            raise RuntimeError("Schema enforcement test did not block schema drift")\n\n        # MERGE/UPSERT proof.\n        first_ids = [r["ticket_id"] for r in df.head(3).to_dict(orient="records")]\n        updates = spark.createDataFrame([(ticket_id, "closed") for ticket_id in first_ids], ["ticket_id", "new_status"])\n        table = DeltaTable.forPath(spark, str(silver_path))\n        (\n            table.alias("target")\n            .merge(updates.alias("source"), "target.ticket_id = source.ticket_id")\n            .whenMatchedUpdate(set={"status": "source.new_status"})\n            .execute()\n        )\n        _write_json(PATHS["quality"] / "delta_merge_evidence.json", {\n            "updated_ticket_ids": first_ids,\n            "operation_history": [row.asDict(recursive=True) for row in table.history(3).collect()],\n        })\n\n        gold_intent = (\n            spark.read.format("delta").load(str(silver_path))\n            .groupBy("category", "intent")\n            .agg(F.count("*").alias("ticket_count"))\n            .orderBy(F.desc("ticket_count"))\n        )\n        gold_channel = (\n            spark.read.format("delta").load(str(silver_path))\n            .groupBy("channel", "priority")\n            .agg(F.count("*").alias("ticket_count"))\n        )\n        intent_path = PATHS["gold"] / "intent_metrics_delta"\n        channel_path = PATHS["gold"] / "channel_metrics_delta"\n        gold_intent.write.format("delta").mode("overwrite").save(str(intent_path))\n        gold_channel.write.format("delta").mode("overwrite").save(str(channel_path))\n        print(f"Silver rows={spark.read.format(\'delta\').load(str(silver_path)).count()}")\n        spark.stop()\n\n\ndef build_rag() -> None:\n    with lineage_run(\n        "build_hybrid_rag_index",\n        ["knowledge_base.kb_chunks"],\n        ["vector_db.novastore_support_kb", "rag.bm25_corpus"],\n    ):\n        import chromadb\n        from sentence_transformers import SentenceTransformer\n\n        chunks_path = PATHS["kb"] / "kb_chunks.jsonl"\n        chunks = _read_jsonl(chunks_path)\n        if not chunks:\n            raise RuntimeError(f"No KB chunks found: {chunks_path}")\n        model = SentenceTransformer(EMBEDDING_MODEL)\n        texts = [item["text"] for item in chunks]\n        vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()\n        client = chromadb.PersistentClient(path=str(PATHS["vector"]))\n        try:\n            client.delete_collection("novastore_support_kb")\n        except Exception:\n            pass\n        collection = client.create_collection("novastore_support_kb", metadata={"hnsw:space": "cosine"})\n        collection.add(\n            ids=[item["chunk_id"] for item in chunks],\n            documents=texts,\n            embeddings=vectors,\n            metadatas=[{\n                "doc_id": item["doc_id"],\n                "title": item["title"],\n                "section": item["section"],\n                "category": item["category"],\n                "source": item["source"],\n            } for item in chunks],\n        )\n        _write_json(PATHS["vector"] / "bm25_corpus.json", chunks)\n        _write_json(PATHS["vector"] / "index_manifest.json", {\n            "collection": "novastore_support_kb",\n            "chunks": len(chunks),\n            "embedding_model": EMBEDDING_MODEL,\n        })\n        print(f"Indexed {collection.count()} chunks")\n\n\ndef _retrieval_components():\n    import chromadb\n    from rank_bm25 import BM25Okapi\n    from sentence_transformers import CrossEncoder, SentenceTransformer\n\n    chunks = json.loads((PATHS["vector"] / "bm25_corpus.json").read_text(encoding="utf-8"))\n    tokenized = [re.findall(r"\\b\\w+\\b", item["text"].lower()) for item in chunks]\n    bm25 = BM25Okapi(tokenized)\n    embedder = SentenceTransformer(EMBEDDING_MODEL)\n    reranker = CrossEncoder(RERANKER_MODEL)\n    client = chromadb.PersistentClient(path=str(PATHS["vector"]))\n    collection = client.get_collection("novastore_support_kb")\n    by_id = {item["chunk_id"]: item for item in chunks}\n    return chunks, by_id, bm25, embedder, reranker, collection\n\n\ndef retrieve(query: str, top_k: int = 5) -> list[dict[str, Any]]:\n    chunks, by_id, bm25, embedder, reranker, collection = _retrieval_components()\n    query_vector = embedder.encode([query], normalize_embeddings=True).tolist()[0]\n    dense = collection.query(query_embeddings=[query_vector], n_results=min(15, len(chunks)))\n    dense_ids = dense["ids"][0]\n\n    sparse_scores = bm25.get_scores(re.findall(r"\\b\\w+\\b", query.lower()))\n    sparse_order = np.argsort(sparse_scores)[::-1][:15]\n    sparse_ids = [chunks[int(i)]["chunk_id"] for i in sparse_order]\n\n    rrf: dict[str, float] = {}\n    for ranking in [dense_ids, sparse_ids]:\n        for rank, chunk_id in enumerate(ranking, start=1):\n            rrf[chunk_id] = rrf.get(chunk_id, 0.0) + 1.0 / (60 + rank)\n    candidates = sorted(rrf, key=rrf.get, reverse=True)[:12]\n    pairs = [(query, by_id[chunk_id]["text"]) for chunk_id in candidates]\n    rerank_scores = reranker.predict(pairs)\n    ranked = sorted(zip(candidates, rerank_scores), key=lambda item: float(item[1]), reverse=True)\n\n    results: list[dict[str, Any]] = []\n    for chunk_id, score in ranked[:top_k]:\n        row = dict(by_id[chunk_id])\n        row["rerank_score"] = float(score)\n        row["rrf_score"] = float(rrf[chunk_id])\n        results.append(row)\n    return results\n\n\ndef evaluate_rag() -> None:\n    with lineage_run(\n        "evaluate_rag_retrieval",\n        ["vector_db.novastore_support_kb", "knowledge_base.test_queries"],\n        ["quality.retrieval_evaluation"],\n    ):\n        tests = pd.read_csv(PATHS["kb"] / "test_queries.csv")\n        rows = []\n        reciprocal_ranks = []\n        for test in tests.to_dict(orient="records"):\n            results = retrieve(test["query"], top_k=5)\n            doc_ids = [item["doc_id"] for item in results]\n            expected = test["expected_doc_id"]\n            rank = doc_ids.index(expected) + 1 if expected in doc_ids else None\n            reciprocal_ranks.append(1.0 / rank if rank else 0.0)\n            rows.append({\n                "query_id": test["query_id"],\n                "query": test["query"],\n                "expected_doc_id": expected,\n                "retrieved_doc_ids": doc_ids,\n                "rank": rank,\n                "hit_at_1": bool(rank == 1),\n                "hit_at_3": bool(rank and rank <= 3),\n            })\n        metrics = {\n            "queries": len(rows),\n            "hit_at_1": sum(row["hit_at_1"] for row in rows) / len(rows),\n            "hit_at_3": sum(row["hit_at_3"] for row in rows) / len(rows),\n            "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks),\n            "details": rows,\n        }\n        _write_json(PATHS["quality"] / "retrieval_evaluation.json", metrics)\n        print(json.dumps({k: v for k, v in metrics.items() if k != "details"}, indent=2))\n\n\ndef publish_answer() -> None:\n    with lineage_run(\n        "publish_grounded_answer",\n        ["vector_db.novastore_support_kb"],\n        ["gold.rag_answers"],\n    ):\n        query = os.getenv(\n            "SUPPORTAI_DEMO_QUERY",\n            "I received a damaged product. What information do you need from me?",\n        )\n        results = retrieve(query, top_k=3)\n        if not results:\n            raise RuntimeError("No RAG context retrieved")\n        answer = " ".join(item["text"] for item in results[:2])\n        payload = {\n            "answer_id": str(uuid.uuid4()),\n            "ticket_id": "DEMO-TICKET-001",\n            "question": query,\n            "answer": answer,\n            "sources": [{\n                "doc_id": item["doc_id"],\n                "source": item["source"],\n                "section": item["section"],\n                "score": item["rerank_score"],\n            } for item in results],\n            "generated_at": _now(),\n            "generation_mode": "extractive-grounded",\n        }\n        _write_json(PATHS["gold"] / "latest_rag_answer.json", payload)\n\n        spark = spark_session("SupportAI_Gold_RAG_Answer")\n        answer_df = pd.DataFrame([{\n            "answer_id": payload["answer_id"],\n            "ticket_id": payload["ticket_id"],\n            "question": payload["question"],\n            "answer": payload["answer"],\n            "source_doc_ids": json.dumps([s["doc_id"] for s in payload["sources"]]),\n            "source_files": json.dumps([s["source"] for s in payload["sources"]]),\n            "generated_at": payload["generated_at"],\n            "generation_mode": payload["generation_mode"],\n        }])\n        spark.createDataFrame(answer_df).write.format("delta").mode("append").save(\n            str(PATHS["gold"] / "rag_answers_delta")\n        )\n        spark.stop()\n        print(json.dumps(payload, indent=2, ensure_ascii=False))\n\n\ndef verify_deliverables() -> None:\n    checks = {\n        "raw_source": PATHS["raw"] / "bitext_support_tickets_enriched.csv",\n        "bronze_delta": PATHS["bronze"] / "tickets_delta" / "_delta_log",\n        "contract_quarantine": PATHS["quarantine"] / "contract_violations.jsonl",\n        "quality_report": PATHS["quality"] / "great_expectations_validation.json",\n        "schema_enforcement": PATHS["quality"] / "delta_schema_enforcement_evidence.json",\n        "merge_evidence": PATHS["quality"] / "delta_merge_evidence.json",\n        "silver_delta": PATHS["silver"] / "tickets_delta" / "_delta_log",\n        "gold_metrics": PATHS["gold"] / "intent_metrics_delta" / "_delta_log",\n        "vector_manifest": PATHS["vector"] / "index_manifest.json",\n        "retrieval_evaluation": PATHS["quality"] / "retrieval_evaluation.json",\n        "rag_answer": PATHS["gold"] / "latest_rag_answer.json",\n        "openlineage_events": PATHS["lineage"] / "openlineage_events.jsonl",\n    }\n    status = {name: path.exists() for name, path in checks.items()}\n    _write_json(PROJECT_ROOT / "deliverables_verification.json", {\n        "passed": all(status.values()),\n        "checks": {name: {"exists": exists, "path": str(checks[name])} for name, exists in status.items()},\n    })\n    for name, exists in status.items():\n        print(f"{\'PASS\' if exists else \'MISS\'}  {name:<24} {checks[name]}")\n    if not all(status.values()):\n        raise RuntimeError("Some required deliverable evidence is missing")\n\n\nCOMMANDS = {\n    "prepare-data": prepare_data,\n    "kafka-producer": kafka_producer,\n    "kafka-consumer-to-bronze": kafka_consumer_to_bronze,\n    "quality-gate": quality_gate,\n    "build-lakehouse": build_lakehouse,\n    "build-rag": build_rag,\n    "evaluate-rag": evaluate_rag,\n    "publish-answer": publish_answer,\n    "verify": verify_deliverables,\n}\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="SupportAI final-project executable stages")\n    parser.add_argument("command", choices=COMMANDS)\n    args = parser.parse_args()\n    print(f"\\n{\'=\' * 78}\\nSUPPORTAI STAGE: {args.command}\\n{\'=\' * 78}")\n    COMMANDS[args.command]()\n    print(f"COMPLETED: {args.command}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

print(f"✅ Real Kafka artifacts generated in: {real_kafka_dir}")
print(f"✅ Executable pipeline generated at: {pipeline_target}")
print("Consumer evidence: SupportTicketContract(**event) -> Bronze Delta / Quarantine")


✅ Real Kafka artifacts generated in: /content/supportai_final_project/optional_real_kafka
✅ Executable pipeline generated at: /content/supportai_final_project/supportai_pipeline.py
Consumer evidence: SupportTicketContract(**event) -> Bronze Delta / Quarantine


## Step 27 — Final Project Summary

This cell prints the exact artifacts and counts that can be shown during the final presentation.

In [32]:
# =====================================================================
# FINAL EXECUTIVE SUMMARY
# =====================================================================

summary = {
    "source_dataset": DATASET_ID,
    "sampled_ticket_events": len(df_tickets),
    "bronze_events": len(df_bronze_stream),
    "silver_valid_unique_tickets": len(df_silver_pd),
    "quarantined_contract_violations": len(df_contract_rejected),
    "knowledge_documents": len(KNOWLEDGE_DOCUMENTS),
    "knowledge_chunks": len(KB_CHUNKS),
    "vector_index_size": vector_collection.count(),
    "great_expectations_passed": ge_passed,
    "dama_quality_passed": dq_report["passed"],
    "retrieval_hit_at_1": round(retrieval_metrics["hit_at_1"], 3),
    "retrieval_hit_at_3": round(retrieval_metrics["hit_at_3"], 3),
    "retrieval_mrr": round(retrieval_metrics["mrr"], 3),
    "delta_schema_enforcement_passed": schema_enforcement_evidence["passed"],
    "official_openlineage_file": str(openlineage_path),
    "executable_airflow_dag": str(dag_path),
    "project_root": str(PROJECT_ROOT.resolve()),
}

summary_path = PROJECT_ROOT / "final_project_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("=" * 78)
print("  SUPPORTAI — FINAL PROJECT COMPLETE")
print("=" * 78)
for key, value in summary.items():
    print(f"{key:<38}: {value}")
print("=" * 78)

  SUPPORTAI — FINAL PROJECT COMPLETE
source_dataset                        : bitext/Bitext-customer-support-llm-chatbot-training-dataset
sampled_ticket_events                 : 200
bronze_events                         : 200
silver_valid_unique_tickets           : 195
quarantined_contract_violations       : 4
knowledge_documents                   : 8
knowledge_chunks                      : 50
vector_index_size                     : 50
great_expectations_passed             : True
dama_quality_passed                   : True
retrieval_hit_at_1                    : 0.9
retrieval_hit_at_3                    : 1.0
retrieval_mrr                         : 0.95
delta_schema_enforcement_passed       : True
official_openlineage_file             : /content/supportai_final_project/lineage_events/openlineage_events.jsonl
executable_airflow_dag                : /content/supportai_final_project/dags/supportai_final_pipeline.py
project_root                          : /content/supportai_final_project
